# Hannah Multi-Regime Volatility Router

This notebook adopts Hannah's multi-regime volatility router LSTM autoencoder and turns the portfolio layer into a multi-regime router. It uses both the dataset benchmark and SPY as context, chooses among score routes with validation/backtest diagnostics, applies explicit beta/concentration controls, logs calibration artifacts, and makes MLflow logging a required final step.

In [1]:
# Bootstrap: works locally, in Colab, or when repo_root is set manually.
import os
import sys
import subprocess
import tempfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DEFAULT_REPO_URL = "https://github.com/halloyy/Portfolio-Optimization-Lib.git"


def _is_repo_root(path: Path) -> bool:
    path = Path(path).expanduser()
    return (path / "pyproject.toml").exists() and (path / "src" / "portfolio_toolkit").exists()


def _candidate_roots() -> list[Path]:
    candidates: list[Path] = []
    if "repo_root" in globals():
        candidates.append(Path(repo_root).expanduser())
    if os.environ.get("PORTFOLIO_REPO_ROOT"):
        candidates.append(Path(os.environ["PORTFOLIO_REPO_ROOT"]).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    candidates.extend(
        [
            Path("/content/Portfolio-Optimization-Lib"),
            Path("/content/Portfolio-Optimizer"),
            Path("/workspace/Portfolio-Optimization-Lib"),
            Path("/Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib"),
        ]
    )
    return candidates


def _find_repo_root() -> Path | None:
    for candidate in _candidate_roots():
        if _is_repo_root(candidate):
            return candidate.resolve()
    return None


repo_root = _find_repo_root()

if repo_root is None and IN_COLAB:
    repo_url = os.environ.get("PORTFOLIO_REPO_URL", DEFAULT_REPO_URL)
    clone_target = Path(os.environ.get("PORTFOLIO_REPO_ROOT", "/content/Portfolio-Optimization-Lib")).expanduser()
    if not clone_target.exists():
        subprocess.run(["git", "clone", repo_url, str(clone_target)], check=True)
    repo_root = clone_target.resolve() if _is_repo_root(clone_target) else None
    if repo_root is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root)], check=True)

if repo_root is None:
    raise RuntimeError(
        "Cannot find repo root. Run this notebook from inside Portfolio-Optimization-Lib, "
        "or set repo_root = Path('/path/to/Portfolio-Optimization-Lib') before this cell."
    )

os.chdir(repo_root)
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio_matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("repo_root =", repo_root)
print("python    =", sys.executable)

repo_root = /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer
python    = /Users/adamthorne/.pyenv/versions/3.12.7/bin/python


In [2]:
import json
import math
from dataclasses import replace
from datetime import date
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.covariance import LedoitWolf

from portfolio_toolkit import (
    PortfolioWeights,
    backtest_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_return_target,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    write_backtest_artifacts,
)
from portfolio_toolkit.config import dataset_identifier

warnings.filterwarnings("ignore")
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

/Users/adamthorne/.pyenv/versions/3.12.7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0 | cuda: False


## Configuration

In [3]:
DATASET_NAME = os.environ.get("DATASET_NAME", "shared_set_2")
MODEL_NAME = "hannah_multiregime_volatility_router_lstm_autoencoder"
RUN_NAME = "Hannah_Multi_Regime_Volatility_Router"
HORIZON = 5
SEQ_LEN = 20
REBALANCE_FREQUENCY = "weekly"
RESEARCH_END = pd.Timestamp(os.environ.get("ROUTER_RESEARCH_END", "2025-12-31"))

LATENT_DIM = 32
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = int(os.environ.get("LSTM_AE_EPOCHS", "18"))
DATES_PER_BATCH = 16
PATIENCE = 5
MASK_RATE = 0.10
GRAD_CLIP = 1.0

# Volatility-first loss profile. Alpha remains present, but risk heads carry more weight than Week 7.
RECON_WEIGHT = 1.00
RETURN_WEIGHT = 0.15
ALPHA_WEIGHT = 1.20
RANK_WEIGHT = 0.20
VOL_WEIGHT = 0.85
DOWNSIDE_WEIGHT = 0.95
TAIL_WEIGHT = 0.70
REGIME_WEIGHT = 0.40
HIGH_RISK_SAMPLE_MULTIPLIER = 1.75

NORMAL_MAX_WEIGHT = 0.10
HIGH_VOL_MAX_WEIGHT_OPTIONS = [0.03, 0.05, 0.075, 0.10]
BASE_TOP_FRACTION = 0.65
HIGH_RISK_MIN_ACTIVE_NAMES = 35
NORMAL_MIN_ACTIVE_NAMES = 18
TURNOVER_BLEND = 0.45
BETA_TARGET = float(os.environ.get("ROUTER_BETA_TARGET", "0.85"))
COV_LOOKBACK_DAYS = 252
MINVAR_TILT_NORMAL = 0.30
MINVAR_TILT_HIGH_RISK = 0.10

DOWNSIDE_SCORE_PENALTY = 0.75
TAIL_SCORE_PENALTY = 1.25
UNCERTAINTY_SCORE_PENALTY = 0.35
REGIME_SCORE_BONUS = 0.08

REGIME_CLASSES = ["calm_risk_on", "high_vol", "drawdown", "rebound"]
REGIME_CLASS_TO_ID = {name: idx for idx, name in enumerate(REGIME_CLASSES)}

FAST_SMOKE = os.environ.get("FAST_SMOKE", "0") == "1"
FULL_ROUTER_GRID = os.environ.get("FULL_ROUTER_GRID", "0") == "1"
# Default to one fixed submission candidate so top-to-bottom runs do not launch the router grid search.
SUBMISSION_MINIMAL_GRID = os.environ.get("SUBMISSION_MINIMAL_GRID", "1") == "1"
MAX_TRAIN_SEQUENCES = int(os.environ.get("MAX_TRAIN_SEQUENCES", "0"))
MAX_VAL_SEQUENCES = int(os.environ.get("MAX_VAL_SEQUENCES", "0"))
BETA_TARGET_GRID = [0.50, 0.70, 0.85, 1.00]
WINNER_TICKERS = ["PLTR", "HOOD", "NVDA", "AVGO", "KTOS", "BWXT"]
RANDOM_SEED = 99
LOG_TO_MLFLOW = os.environ.get("SKIP_MLFLOW", "0") != "1"
MLFLOW_REQUIRED = os.environ.get("MLFLOW_REQUIRED", "1") == "1"

output_dir = repo_root / "runs" / f"supervised_lstm_autoencoder_multiregime_router_{DATASET_NAME}"
output_dir.mkdir(parents=True, exist_ok=True)
NOTEBOOK_PATH = repo_root / "MODELS" / "Hannah" / "supervised_lstm_autoencoder_multiregime_router.ipynb"

spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)
research_end_tag = RESEARCH_END.strftime("%Y%m%d")
research_spec = replace(
    spec,
    name=f"{spec.name}_through_{research_end_tag}",
    end_date=RESEARCH_END.date(),
    test_end=RESEARCH_END.date(),
    dataset_id=f"{DATASET_NAME}_through_{research_end_tag}",
)
BENCHMARK_CONTEXT_TICKER = research_spec.benchmark_ticker.upper()
tradable_tickers = [ticker for ticker in research_spec.tickers if ticker != research_spec.benchmark_ticker]
backtest_spec = replace(research_spec, tickers=tradable_tickers)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True

print("Dataset preset:", DATASET_NAME)
print("Research end cap:", RESEARCH_END.date())
print("Research dataset id:", dataset_identifier(research_spec, repo_root=repo_root))
print("Tradable tickers:", len(tradable_tickers))
print("Benchmark ticker:", research_spec.benchmark_ticker)
print("Benchmark context ticker:", BENCHMARK_CONTEXT_TICKER)
print("Train/Val/Test windows:", research_spec.train_start, research_spec.train_end, research_spec.val_start, research_spec.val_end, research_spec.test_start, research_spec.test_end)
print("Epochs:", EPOCHS, "| FAST_SMOKE:", FAST_SMOKE, "| MLflow logging:", LOG_TO_MLFLOW, "| MLflow required:", MLFLOW_REQUIRED)
print("Sequence caps:", {"train": MAX_TRAIN_SEQUENCES or None, "val": MAX_VAL_SEQUENCES or None})
print("Submission minimal grid:", SUBMISSION_MINIMAL_GRID)

Dataset preset: shared_set_2
Research end cap: 2025-12-31
Research dataset id: shared_set_2_through_20251231
Tradable tickers: 25
Benchmark ticker: SPY
Benchmark context ticker: SPY
Train/Val/Test windows: 2014-01-02 2019-12-31 2020-01-02 2021-12-31 2022-01-03 2025-12-31
Epochs: 18 | FAST_SMOKE: False | MLflow logging: True | MLflow required: True
Sequence caps: {'train': None, 'val': None}
Submission minimal grid: True


## Volatility And Stress Features

In [4]:
BASE_FEATURE_NAMES = [
    "return_1d", "return_5d", "return_20d", "return_60d",
    "vol_5d", "vol_20d", "vol_60d", "downside_vol_20d", "atr_14",
    "momentum_5d", "momentum_20d", "momentum_60d", "momentum_120d",
    "rsi_14", "macd_hist", "bollinger_z_20d",
    "price_to_sma_20d", "price_to_sma_50d", "price_to_sma_200d",
    "volume_zscore_20d", "dollar_volume_ratio_20d", "intraday_range",
    "beta_20d_spy", "beta_60d_spy",
    "excess_return_20d_vs_spy", "excess_return_60d_vs_spy", "relative_momentum_20d_vs_spy",
]

CUSTOM_FEATURE_NAMES = [
    "mom_vol_ratio_60d", "trend_spread_20_50", "trend_spread_50_200", "rolling_sharpe_60d", "drawdown_60d",
    "spy_vol_20d_ann", "spy_momentum_20d", "spy_momentum_60d", "spy_drawdown_60d", "market_breadth_20d",
    "pct_above_sma50", "pct_above_sma200", "pct_20d_highs", "pct_20d_lows", "avg_universe_drawdown_60d",
    "cross_sectional_dispersion_20d", "gap_frequency_20d", "volume_shock_20d", "abnormal_range_20d",
    "vol_compression_ratio_20_60", "distance_from_60d_high", "momentum_x_spy_vol", "momentum_x_spy_drawdown",
    "beta_instability_60d",
    "avg_pairwise_corr_20d", "avg_pairwise_corr_60d", "corr_breakdown_20_60",
    "breadth_deterioration_20d", "dollar_volume_shock_20d", "abnormal_range_persistence_10d",
    "liquidity_stress_20d", "benchmark_agnostic_stress", "beta_x_corr_stress",
    "bench_vol_20d_ann", "bench_momentum_20d", "bench_momentum_60d", "bench_drawdown_60d",
    "bench_sma_distance_50d", "bench_rebound_state", "bench_stress_state", "beta_60d_benchmark",
    "dual_context_vol_spread", "benchmark_spy_momentum_spread", "rebound_breadth_confirmation",
]

ALL_FEATURE_NAMES = BASE_FEATURE_NAMES + CUSTOM_FEATURE_NAMES
REGIME_CONTEXT_COLUMNS = ["spy_vol_20d_ann", "spy_momentum_20d", "spy_momentum_60d", "spy_drawdown_60d", "bench_vol_20d_ann", "bench_momentum_20d", "bench_momentum_60d", "bench_drawdown_60d", "bench_rebound_state", "bench_stress_state"]


def _safe_pct_change(series: pd.Series, periods: int = 1) -> pd.Series:
    return series.astype(float).pct_change(periods)


def _rolling_average_pairwise_corr(return_wide: pd.DataFrame, window: int, min_periods: int) -> pd.Series:
    rows = []
    index = return_wide.index
    values = return_wide.to_numpy(dtype=float)
    for end in range(len(index)):
        start = max(0, end - window + 1)
        block = values[start : end + 1]
        valid_cols = np.isfinite(block).sum(axis=0) >= min_periods
        if valid_cols.sum() < 2:
            rows.append(np.nan)
            continue
        corr = np.corrcoef(block[:, valid_cols], rowvar=False)
        if corr.ndim != 2:
            rows.append(np.nan)
            continue
        mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
        rows.append(float(np.nanmean(corr[mask])))
    return pd.Series(rows, index=index)



def _select_context_ticker(panel: pd.DataFrame, preferred: str, fallback: str = "SPY") -> str:
    available = set(panel["ticker"].astype(str).str.upper())
    if preferred.upper() in available:
        return preferred.upper()
    if fallback.upper() in available:
        return fallback.upper()
    return sorted(available)[0]


def _make_context_features(panel: pd.DataFrame, ticker: str, prefix: str) -> pd.DataFrame:
    context = panel.loc[panel["ticker"] == ticker, ["date", "adj_close"]].sort_values("date").copy()
    if context.empty:
        return pd.DataFrame({"date": pd.to_datetime(panel["date"].drop_duplicates())})
    ret = context["adj_close"].pct_change()
    out = context.loc[:, ["date"]].copy()
    out[f"{prefix}_vol_20d_ann"] = ret.rolling(20, min_periods=20).std(ddof=0) * np.sqrt(252.0)
    out[f"{prefix}_momentum_20d"] = context["adj_close"].pct_change(20)
    out[f"{prefix}_momentum_60d"] = context["adj_close"].pct_change(60)
    high_60 = context["adj_close"].rolling(60, min_periods=20).max()
    sma_50 = context["adj_close"].rolling(50, min_periods=30).mean()
    out[f"{prefix}_drawdown_60d"] = context["adj_close"] / high_60.replace(0.0, np.nan) - 1.0
    out[f"{prefix}_sma_distance_50d"] = context["adj_close"] / sma_50.replace(0.0, np.nan) - 1.0
    out[f"{prefix}_rebound_state"] = ((out[f"{prefix}_drawdown_60d"] < -0.08) & (out[f"{prefix}_momentum_20d"] > 0.0)).astype(float)
    out[f"{prefix}_stress_state"] = ((out[f"{prefix}_vol_20d_ann"] > out[f"{prefix}_vol_20d_ann"].rolling(252, min_periods=60).quantile(0.70)) | (out[f"{prefix}_drawdown_60d"] < -0.12)).astype(float)
    return out


def _rolling_context_beta(panel: pd.DataFrame, context_ticker: str, out_col: str, window: int = 60) -> pd.DataFrame:
    context = panel.loc[panel["ticker"] == context_ticker, ["date", "adj_close"]].sort_values("date").copy()
    context["context_return"] = context["adj_close"].pct_change()
    temp = panel.loc[:, ["date", "ticker", "adj_close"]].sort_values(["ticker", "date"]).copy()
    temp["asset_return"] = temp.groupby("ticker", sort=False)["adj_close"].pct_change()
    temp = temp.merge(context.loc[:, ["date", "context_return"]], on="date", how="left")
    pieces = []
    for ticker, group in temp.groupby("ticker", sort=False):
        cov = group["asset_return"].rolling(window, min_periods=30).cov(group["context_return"])
        var = group["context_return"].rolling(window, min_periods=30).var()
        pieces.append(pd.DataFrame({"date": group["date"].to_numpy(), "ticker": ticker, out_col: (cov / var.replace(0.0, np.nan)).to_numpy()}))
    return pd.concat(pieces, ignore_index=True)


def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    """Rebuild the exact multi-regime router LSTM-AE feature table from raw long-form prices."""
    frame = build_features(prices, feature_names=BASE_FEATURE_NAMES)
    panel = prices.copy()
    panel["date"] = pd.to_datetime(panel["date"], utc=True).dt.tz_localize(None)
    panel["ticker"] = panel["ticker"].astype(str).str.upper()
    panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
    eps = 1e-6

    frame["mom_vol_ratio_60d"] = frame["momentum_60d"] / (frame["vol_60d"].abs() + eps)
    frame["trend_spread_20_50"] = frame["price_to_sma_20d"] - frame["price_to_sma_50d"]
    frame["trend_spread_50_200"] = frame["price_to_sma_50d"] - frame["price_to_sma_200d"]
    frame["rolling_sharpe_60d"] = frame["return_60d"] / (frame["vol_60d"].abs() * np.sqrt(60.0) + eps)

    grouped_close = panel.groupby("ticker", sort=False)["adj_close"]
    rolling_high_20 = grouped_close.transform(lambda s: s.rolling(20, min_periods=20).max())
    rolling_low_20 = grouped_close.transform(lambda s: s.rolling(20, min_periods=20).min())
    rolling_high_60 = grouped_close.transform(lambda s: s.rolling(60, min_periods=20).max())
    prev_close = grouped_close.shift(1)

    event = panel.loc[:, ["date", "ticker"]].copy()
    event["drawdown_60d"] = panel["adj_close"] / rolling_high_60.replace(0.0, np.nan) - 1.0
    event["distance_from_60d_high"] = panel["adj_close"] / rolling_high_60.replace(0.0, np.nan) - 1.0
    event["gap_abs"] = (panel["open"] / prev_close.replace(0.0, np.nan) - 1.0).abs()
    event["range_ratio"] = (panel["high"] - panel["low"]) / panel["adj_close"].replace(0.0, np.nan)
    event["gap_frequency_20d"] = event.groupby(panel["ticker"], sort=False)["gap_abs"].transform(lambda s: (s > 0.03).rolling(20, min_periods=10).mean())
    event["abnormal_range_20d"] = event["range_ratio"] / (event.groupby(panel["ticker"], sort=False)["range_ratio"].transform(lambda s: s.rolling(20, min_periods=10).mean()) + eps)
    event["abnormal_range_persistence_10d"] = event.groupby(panel["ticker"], sort=False)["abnormal_range_20d"].transform(lambda s: s.rolling(10, min_periods=5).mean())
    rolling_vol_20 = grouped_close.transform(lambda s: _safe_pct_change(s).rolling(20, min_periods=20).std(ddof=0))
    rolling_vol_60 = grouped_close.transform(lambda s: _safe_pct_change(s).rolling(60, min_periods=30).std(ddof=0))
    event["vol_compression_ratio_20_60"] = rolling_vol_20 / (rolling_vol_60 + eps)
    dollar_volume = panel["adj_close"] * panel["volume"]
    dollar_volume_ma_20 = dollar_volume.groupby(panel["ticker"], sort=False).transform(lambda s: s.rolling(20, min_periods=10).mean())
    volume_ma_20 = panel.groupby("ticker", sort=False)["volume"].transform(lambda s: s.rolling(20, min_periods=10).mean())
    event["volume_shock_20d"] = panel["volume"] / volume_ma_20.replace(0.0, np.nan)
    event["dollar_volume_shock_20d"] = dollar_volume / dollar_volume_ma_20.replace(0.0, np.nan)
    event["liquidity_stress_20d"] = event["abnormal_range_persistence_10d"] / (event["dollar_volume_shock_20d"].clip(lower=0.10) + eps)
    event = event.drop(columns=["gap_abs", "range_ratio"])
    frame = frame.merge(event, on=["date", "ticker"], how="left")

    spy = panel.loc[panel["ticker"] == "SPY", ["date", "adj_close"]].sort_values("date").copy()
    if spy.empty:
        regime = pd.DataFrame({"date": pd.to_datetime(panel["date"].drop_duplicates())})
        for col in REGIME_CONTEXT_COLUMNS:
            regime[col] = np.nan
    else:
        spy_ret = spy["adj_close"].pct_change()
        regime = spy.loc[:, ["date"]].copy()
        regime["spy_vol_20d_ann"] = spy_ret.rolling(20, min_periods=20).std(ddof=0) * np.sqrt(252.0)
        regime["spy_momentum_20d"] = spy["adj_close"].pct_change(20)
        regime["spy_momentum_60d"] = spy["adj_close"].pct_change(60)
        spy_high_60 = spy["adj_close"].rolling(60, min_periods=20).max()
        regime["spy_drawdown_60d"] = spy["adj_close"] / spy_high_60.replace(0.0, np.nan) - 1.0
    frame = frame.merge(regime, on="date", how="left")

    non_benchmark = panel.loc[panel["ticker"] != "SPY"].copy()
    nb_close = non_benchmark.groupby("ticker", sort=False)["adj_close"]
    nb_return = nb_close.pct_change()
    sma_20 = nb_close.transform(lambda s: s.rolling(20, min_periods=20).mean())
    sma_50 = nb_close.transform(lambda s: s.rolling(50, min_periods=30).mean())
    sma_200 = nb_close.transform(lambda s: s.rolling(200, min_periods=100).mean())
    high_20 = nb_close.transform(lambda s: s.rolling(20, min_periods=20).max())
    low_20 = nb_close.transform(lambda s: s.rolling(20, min_periods=20).min())
    high_60_nb = nb_close.transform(lambda s: s.rolling(60, min_periods=20).max())
    breadth_panel = non_benchmark.loc[:, ["date", "ticker"]].copy()
    breadth_panel["above_sma20"] = non_benchmark["adj_close"] > sma_20
    breadth_panel["above_sma50"] = non_benchmark["adj_close"] > sma_50
    breadth_panel["above_sma200"] = non_benchmark["adj_close"] > sma_200
    breadth_panel["is_20d_high"] = non_benchmark["adj_close"] >= high_20
    breadth_panel["is_20d_low"] = non_benchmark["adj_close"] <= low_20
    breadth_panel["universe_drawdown_60d"] = non_benchmark["adj_close"] / high_60_nb.replace(0.0, np.nan) - 1.0
    breadth_panel["daily_return"] = nb_return.to_numpy(dtype=float)

    breadth = breadth_panel.groupby("date").agg(
        market_breadth_20d=("above_sma20", "mean"),
        pct_above_sma50=("above_sma50", "mean"),
        pct_above_sma200=("above_sma200", "mean"),
        pct_20d_highs=("is_20d_high", "mean"),
        pct_20d_lows=("is_20d_low", "mean"),
        avg_universe_drawdown_60d=("universe_drawdown_60d", "mean"),
        cross_sectional_dispersion_20d=("daily_return", "std"),
    ).reset_index()
    breadth["cross_sectional_dispersion_20d"] = breadth["cross_sectional_dispersion_20d"].rolling(20, min_periods=10).mean()
    breadth["breadth_deterioration_20d"] = breadth["market_breadth_20d"] - breadth["market_breadth_20d"].shift(20)

    ret_wide = non_benchmark.pivot(index="date", columns="ticker", values="adj_close").sort_index().pct_change()
    corr = pd.DataFrame({
        "date": ret_wide.index,
        "avg_pairwise_corr_20d": _rolling_average_pairwise_corr(ret_wide, 20, 12).to_numpy(dtype=float),
        "avg_pairwise_corr_60d": _rolling_average_pairwise_corr(ret_wide, 60, 30).to_numpy(dtype=float),
    })
    corr["corr_breakdown_20_60"] = corr["avg_pairwise_corr_20d"] - corr["avg_pairwise_corr_60d"]
    breadth = breadth.merge(corr, on="date", how="left")
    breadth["benchmark_agnostic_stress"] = (
        breadth["cross_sectional_dispersion_20d"].rank(pct=True)
        + (-breadth["avg_universe_drawdown_60d"]).rank(pct=True)
        + breadth["avg_pairwise_corr_20d"].rank(pct=True)
        + (-breadth["breadth_deterioration_20d"]).rank(pct=True)
    ) / 4.0
    frame = frame.merge(breadth, on="date", how="left")

    frame["momentum_x_spy_vol"] = frame["momentum_20d"] * frame["spy_vol_20d_ann"]
    frame["momentum_x_spy_drawdown"] = frame["momentum_20d"] * frame["spy_drawdown_60d"]
    frame["beta_instability_60d"] = frame.groupby("ticker", sort=False)["beta_20d_spy"].transform(lambda s: s.rolling(60, min_periods=20).std(ddof=0))
    frame["beta_x_corr_stress"] = frame["beta_60d_spy"].abs() * frame["avg_pairwise_corr_20d"]

    # Dual benchmark context. On secret_set_3, the dataset benchmark is used as the primary benchmark context.
    # On local datasets without dataset benchmark, the bench-prefixed fields fall back to SPY so the notebook remains runnable.
    benchmark_context_ticker = _select_context_ticker(panel, BENCHMARK_CONTEXT_TICKER, fallback="SPY")
    benchmark_context = _make_context_features(panel, benchmark_context_ticker, "bench")
    frame = frame.merge(benchmark_context, on="date", how="left")
    beta_benchmark = _rolling_context_beta(panel, benchmark_context_ticker, "beta_60d_benchmark")
    frame = frame.merge(beta_benchmark, on=["date", "ticker"], how="left")
    frame["dual_context_vol_spread"] = frame["bench_vol_20d_ann"] - frame["spy_vol_20d_ann"]
    frame["benchmark_spy_momentum_spread"] = frame["bench_momentum_20d"] - frame["spy_momentum_20d"]
    frame["rebound_breadth_confirmation"] = frame["bench_rebound_state"] * (frame["breadth_deterioration_20d"] > 0).astype(float)


    return frame.loc[:, ["date", "ticker"] + ALL_FEATURE_NAMES]

In [5]:
prices_all = load_prices(DATASET_NAME, repo_root=repo_root)
prices_all["date"] = pd.to_datetime(prices_all["date"], utc=True).dt.tz_localize(None)
prices = prices_all.loc[prices_all["date"] <= RESEARCH_END].copy()

# Make the standard toolkit backtester use the pre-2023 research data instead of the full cached dataset.
research_cache_path = repo_root / "data_cache" / f"{dataset_identifier(research_spec, repo_root=repo_root)}.parquet"
research_cache_path.parent.mkdir(parents=True, exist_ok=True)
prices.to_parquet(research_cache_path, index=False)
print("Wrote research cache:", research_cache_path)
print("Price frame shape:", prices.shape)
print("Date range:", prices["date"].min(), "->", prices["date"].max())
print("Unique tickers:", prices["ticker"].nunique())

feature_frame = build_model_features(prices)
print("Feature frame shape:", feature_frame.shape)
print("Feature count:", len(ALL_FEATURE_NAMES))
display(feature_frame.head())

Wrote research cache: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/data_cache/shared_set_2_through_20251231.parquet
Price frame shape: (78468, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00
Unique tickers: 26
Feature frame shape: (78468, 73)
Feature count: 71


,date,ticker,return_1d,return_5d,return_20d,return_60d,vol_5d,vol_20d,vol_60d,downside_vol_20d,...,bench_momentum_20d,bench_momentum_60d,bench_drawdown_60d,bench_sma_distance_50d,bench_rebound_state,bench_stress_state,beta_60d_benchmark,dual_context_vol_spread,benchmark_spy_momentum_spread,rebound_breadth_confirmation
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0
1,2014-01-03,AAPL,-0.021966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0
2,2014-01-06,AAPL,0.005453,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0
3,2014-01-07,AAPL,-0.007152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0
4,2014-01-08,AAPL,0.006333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0


## Tail Targets, Regimes, And Sequences

In [6]:
return_target_col = f"forward_return_{HORIZON}d"
alpha_target_col = f"forward_alpha_{HORIZON}d_vs_spy"
forward_vol_target_col = f"forward_realized_vol_{HORIZON}d"
bottom_quintile_col = f"bottom_return_quintile_{HORIZON}d"
tail_loss_target_col = f"forward_tail_loss_{HORIZON}d"


def make_forward_risk_targets(prices: pd.DataFrame, horizon: int) -> pd.DataFrame:
    panel = prices.copy()
    panel["date"] = pd.to_datetime(panel["date"], utc=True).dt.tz_localize(None)
    panel["ticker"] = panel["ticker"].astype(str).str.upper()
    panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
    panel["daily_return"] = panel.groupby("ticker", sort=False)["adj_close"].pct_change()

    def _future_vol(ret: pd.Series) -> pd.Series:
        shifted = ret.shift(-1)
        return shifted.iloc[::-1].rolling(horizon, min_periods=horizon).std(ddof=0).iloc[::-1] * np.sqrt(252.0)

    def _future_tail_loss(ret: pd.Series) -> pd.Series:
        shifted = ret.shift(-1)
        worst = shifted.iloc[::-1].rolling(horizon, min_periods=horizon).min().iloc[::-1]
        return (-worst).clip(lower=0.0)

    panel[forward_vol_target_col] = panel.groupby("ticker", sort=False)["daily_return"].transform(_future_vol)
    panel[tail_loss_target_col] = panel.groupby("ticker", sort=False)["daily_return"].transform(_future_tail_loss)
    return panel.loc[:, ["date", "ticker", forward_vol_target_col, tail_loss_target_col]]


def make_regime_thresholds(feature_frame: pd.DataFrame, train_start, train_end) -> dict[str, float]:
    context_cols = ["date", *REGIME_CONTEXT_COLUMNS, "benchmark_agnostic_stress", "breadth_deterioration_20d", "dual_context_vol_spread", "benchmark_spy_momentum_spread"]
    context = feature_frame.loc[:, context_cols].drop_duplicates("date").copy()
    dates = pd.to_datetime(context["date"])
    train_context = context.loc[(dates >= pd.Timestamp(train_start)) & (dates <= pd.Timestamp(train_end))].replace([np.inf, -np.inf], np.nan)
    return {
        "high_vol_spy_vol_20d_ann": float(train_context["spy_vol_20d_ann"].quantile(0.70)),
        "drawdown_spy_drawdown_60d": float(min(-0.04, train_context["spy_drawdown_60d"].quantile(0.25))),
        "rebound_spy_momentum_20d": float(max(0.0, train_context["spy_momentum_20d"].median())),
        "stress_benchmark_agnostic": float(train_context["benchmark_agnostic_stress"].quantile(0.70)),
        "breadth_deterioration_20d": float(train_context["breadth_deterioration_20d"].quantile(0.25)),
        "bench_high_vol_20d_ann": float(train_context["bench_vol_20d_ann"].quantile(0.70)),
        "bench_drawdown_60d": float(min(-0.06, train_context["bench_drawdown_60d"].quantile(0.25))),
        "dual_context_vol_spread": float(train_context["dual_context_vol_spread"].quantile(0.70)),
    }


def assign_market_regime(frame: pd.DataFrame, thresholds: dict[str, float]) -> pd.Series:
    vol = pd.to_numeric(frame["spy_vol_20d_ann"], errors="coerce")
    drawdown = pd.to_numeric(frame["spy_drawdown_60d"], errors="coerce")
    momentum = pd.to_numeric(frame["spy_momentum_20d"], errors="coerce")
    agnostic_stress = pd.to_numeric(frame["benchmark_agnostic_stress"], errors="coerce")
    breadth_drop = pd.to_numeric(frame["breadth_deterioration_20d"], errors="coerce")
    labels = pd.Series("calm_risk_on", index=frame.index, dtype="object")
    bench_vol = pd.to_numeric(frame["bench_vol_20d_ann"], errors="coerce")
    bench_drawdown = pd.to_numeric(frame["bench_drawdown_60d"], errors="coerce")
    vol_spread = pd.to_numeric(frame["dual_context_vol_spread"], errors="coerce")
    labels.loc[(vol >= thresholds["high_vol_spy_vol_20d_ann"]) | (bench_vol >= thresholds["bench_high_vol_20d_ann"]) | (vol_spread >= thresholds["dual_context_vol_spread"]) | (agnostic_stress >= thresholds["stress_benchmark_agnostic"])] = "high_vol"
    labels.loc[(drawdown <= thresholds["drawdown_spy_drawdown_60d"]) | (bench_drawdown <= thresholds["bench_drawdown_60d"]) | (breadth_drop <= thresholds["breadth_deterioration_20d"])] = "drawdown"
    rebound = (drawdown <= thresholds["drawdown_spy_drawdown_60d"]) & (momentum > thresholds["rebound_spy_momentum_20d"])
    labels.loc[rebound] = "rebound"
    return labels


return_targets = make_forward_return_target(prices, horizon=HORIZON)
alpha_targets = make_forward_alpha_target(prices, horizon=HORIZON)
risk_targets = make_forward_risk_targets(prices, horizon=HORIZON)
regime_thresholds = make_regime_thresholds(feature_frame, research_spec.train_start, research_spec.train_end)

feature_frame = feature_frame.copy()
feature_frame["market_regime"] = assign_market_regime(feature_frame, regime_thresholds)
feature_frame["market_regime_code"] = feature_frame["market_regime"].map(REGIME_CLASS_TO_ID).astype(int)
feature_frame["is_high_risk_regime"] = feature_frame["market_regime"].isin(["high_vol", "drawdown"]).astype(float)

target_frame = feature_frame.merge(return_targets, on=["date", "ticker"], how="left")
target_frame = target_frame.merge(alpha_targets, on=["date", "ticker"], how="left")
target_frame = target_frame.merge(risk_targets, on=["date", "ticker"], how="left")
target_frame = target_frame.replace([np.inf, -np.inf], np.nan)
target_frame[bottom_quintile_col] = (target_frame.groupby("date")[return_target_col].rank(method="average", pct=True) <= 0.20).astype(float)
target_frame = (
    target_frame.dropna(subset=ALL_FEATURE_NAMES + [return_target_col, alpha_target_col, forward_vol_target_col, tail_loss_target_col])
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)
target_frame["historical_vol_20d"] = np.clip(target_frame["vol_20d"].to_numpy(dtype=float) * np.sqrt(252.0), 1e-4, None)
target_frame[forward_vol_target_col] = np.clip(target_frame[forward_vol_target_col].to_numpy(dtype=float), 1e-4, None)
target_frame[tail_loss_target_col] = np.clip(target_frame[tail_loss_target_col].to_numpy(dtype=float), 0.0, None)
target_frame["forward_vol_log"] = np.log1p(target_frame[forward_vol_target_col])
target_frame["tail_loss_log"] = np.log1p(target_frame[tail_loss_target_col])


def _split_mask(frame: pd.DataFrame, start, end) -> pd.Series:
    dates = pd.to_datetime(frame["date"])
    return (dates >= pd.Timestamp(start)) & (dates <= pd.Timestamp(end))


train_rows = target_frame.loc[_split_mask(target_frame, research_spec.train_start, research_spec.train_end)].copy()
val_rows = target_frame.loc[_split_mask(target_frame, research_spec.val_start, research_spec.val_end)].copy()
test_rows = target_frame.loc[_split_mask(target_frame, research_spec.test_start, research_spec.test_end)].copy()
train_model_rows = train_rows.loc[train_rows["ticker"] != research_spec.benchmark_ticker].copy()

train_means = train_model_rows[ALL_FEATURE_NAMES].mean()
train_stds = train_model_rows[ALL_FEATURE_NAMES].std(ddof=0).replace(0.0, 1.0)

target_means = {
    "return": float(train_model_rows[return_target_col].mean()),
    "alpha": float(train_model_rows[alpha_target_col].mean()),
    "vol_log": float(train_model_rows["forward_vol_log"].mean()),
    "tail_log": float(train_model_rows["tail_loss_log"].mean()),
}
target_stds = {
    "return": float(train_model_rows[return_target_col].std(ddof=0) or 1.0),
    "alpha": float(train_model_rows[alpha_target_col].std(ddof=0) or 1.0),
    "vol_log": float(train_model_rows["forward_vol_log"].std(ddof=0) or 1.0),
    "tail_log": float(train_model_rows["tail_loss_log"].std(ddof=0) or 1.0),
}


def standardize_features(frame: pd.DataFrame) -> np.ndarray:
    return ((frame[ALL_FEATURE_NAMES] - train_means) / train_stds).to_numpy(dtype=np.float32)


X_all = standardize_features(target_frame)

print("Modeling frame:", target_frame.shape)
print("Train/Val/Test rows:", len(train_rows), len(val_rows), len(test_rows))
print("Train rows excluding benchmark:", len(train_model_rows))
print("Feature count:", len(ALL_FEATURE_NAMES))
print("Regime thresholds:", json.dumps(regime_thresholds, indent=2, sort_keys=True))
print("Regime distribution by unique date:")
display(target_frame.drop_duplicates(["date"])["market_regime"].value_counts(normalize=True).rename("share").to_frame())

Modeling frame: (73151, 84)
Train/Val/Test rows: 34081 13130 25940
Train rows excluding benchmark: 32770
Feature count: 71
Regime thresholds: {
  "bench_drawdown_60d": -0.06,
  "bench_high_vol_20d_ann": 0.1322621414637131,
  "breadth_deterioration_20d": -0.28,
  "drawdown_spy_drawdown_60d": -0.04,
  "dual_context_vol_spread": 0.0,
  "high_vol_spy_vol_20d_ann": 0.1322621414637131,
  "rebound_spy_momentum_20d": 0.012521517306339325,
  "stress_benchmark_agnostic": 0.5302323356942296
}
Regime distribution by unique date:


,share
market_regime,
high_vol,0.602345
drawdown,0.348969
rebound,0.048685


In [7]:
def make_model_sequences(frame: pd.DataFrame, X: np.ndarray, seq_len: int) -> tuple[np.ndarray, pd.DataFrame]:
    Xs: list[np.ndarray] = []
    meta: list[dict[str, object]] = []
    for ticker, group in frame.groupby("ticker", sort=False):
        positions = group.index.to_numpy(dtype=int)
        if len(positions) < seq_len:
            continue
        dates = pd.to_datetime(group["date"]).to_numpy()
        for end_offset in range(seq_len - 1, len(positions)):
            window_positions = positions[end_offset - seq_len + 1 : end_offset + 1]
            row = group.iloc[end_offset]
            Xs.append(X[window_positions])
            meta.append(
                {
                    "date": pd.Timestamp(dates[end_offset]),
                    "ticker": ticker,
                    "row_position": int(positions[end_offset]),
                    "target_return": float(row[return_target_col]),
                    "target_alpha": float(row[alpha_target_col]),
                    "target_forward_vol": float(row[forward_vol_target_col]),
                    "target_tail_loss": float(row[tail_loss_target_col]),
                    "target_vol_log": float(row["forward_vol_log"]),
                    "target_tail_log": float(row["tail_loss_log"]),
                    "target_bottom_quintile": float(row[bottom_quintile_col]),
                    "market_regime": str(row["market_regime"]),
                    "market_regime_code": int(row["market_regime_code"]),
                    "is_high_risk_regime": float(row["is_high_risk_regime"]),
                    "historical_vol_20d": float(row["historical_vol_20d"]),
                    "spy_vol_20d_ann": float(row["spy_vol_20d_ann"]),
                    "spy_drawdown_60d": float(row["spy_drawdown_60d"]),
                    "spy_momentum_20d": float(row["spy_momentum_20d"]),
                    "beta_60d_spy": float(row["beta_60d_spy"]),
                    "benchmark_agnostic_stress": float(row["benchmark_agnostic_stress"]),
                    "avg_pairwise_corr_20d": float(row["avg_pairwise_corr_20d"]),
                    "bench_vol_20d_ann": float(row["bench_vol_20d_ann"]),
                    "bench_drawdown_60d": float(row["bench_drawdown_60d"]),
                    "bench_momentum_20d": float(row["bench_momentum_20d"]),
                    "bench_rebound_state": float(row["bench_rebound_state"]),
                    "bench_stress_state": float(row["bench_stress_state"]),
                    "beta_60d_benchmark": float(row["beta_60d_benchmark"]),
                    "dual_context_vol_spread": float(row["dual_context_vol_spread"]),
                    "rebound_breadth_confirmation": float(row["rebound_breadth_confirmation"]),
                }
            )
    return np.stack(Xs).astype(np.float32), pd.DataFrame(meta)


X_seq, seq_meta = make_model_sequences(target_frame, X_all, SEQ_LEN)
seq_meta["return_scaled"] = ((seq_meta["target_return"] - target_means["return"]) / target_stds["return"]).astype(np.float32)
seq_meta["alpha_scaled"] = ((seq_meta["target_alpha"] - target_means["alpha"]) / target_stds["alpha"]).astype(np.float32)
seq_meta["vol_scaled"] = ((seq_meta["target_vol_log"] - target_means["vol_log"]) / target_stds["vol_log"]).astype(np.float32)
seq_meta["tail_scaled"] = ((seq_meta["target_tail_log"] - target_means["tail_log"]) / target_stds["tail_log"]).astype(np.float32)
seq_meta["sample_weight"] = 1.0 + (HIGH_RISK_SAMPLE_MULTIPLIER - 1.0) * seq_meta["is_high_risk_regime"].astype(float)

non_benchmark_seq = seq_meta["ticker"] != research_spec.benchmark_ticker
train_seq_mask = _split_mask(seq_meta, research_spec.train_start, research_spec.train_end) & non_benchmark_seq
val_seq_mask = _split_mask(seq_meta, research_spec.val_start, research_spec.val_end) & non_benchmark_seq
test_seq_mask = _split_mask(seq_meta, research_spec.test_start, research_spec.test_end) & non_benchmark_seq

X_train_seq = X_seq[train_seq_mask.to_numpy()]
X_val_seq = X_seq[val_seq_mask.to_numpy()]
X_test_seq_all = X_seq[test_seq_mask.to_numpy()]

meta_train = seq_meta.loc[train_seq_mask].reset_index(drop=True)
meta_val = seq_meta.loc[val_seq_mask].reset_index(drop=True)
meta_test_all = seq_meta.loc[test_seq_mask].reset_index(drop=True)


def cap_sequence_split_by_date(X_split: np.ndarray, meta_split: pd.DataFrame, max_sequences: int, seed: int) -> tuple[np.ndarray, pd.DataFrame]:
    if max_sequences <= 0 or len(meta_split) <= max_sequences:
        return X_split, meta_split
    unique_dates = pd.DatetimeIndex(pd.to_datetime(meta_split["date"]).drop_duplicates()).sort_values()
    if len(unique_dates) == 0:
        return X_split, meta_split
    avg_rows_per_date = max(len(meta_split) / len(unique_dates), 1.0)
    n_dates = max(1, min(len(unique_dates), int(np.ceil(max_sequences / avg_rows_per_date))))
    rng = np.random.default_rng(seed)
    selected_dates = pd.DatetimeIndex(rng.choice(unique_dates.to_numpy(), size=n_dates, replace=False)).sort_values()
    mask = pd.to_datetime(meta_split["date"]).isin(selected_dates)
    capped_meta = meta_split.loc[mask].reset_index(drop=True)
    capped_X = X_split[mask.to_numpy()]
    return capped_X, capped_meta


X_train_seq, meta_train = cap_sequence_split_by_date(X_train_seq, meta_train, MAX_TRAIN_SEQUENCES, RANDOM_SEED)
X_val_seq, meta_val = cap_sequence_split_by_date(X_val_seq, meta_val, MAX_VAL_SEQUENCES, RANDOM_SEED + 1)

y_train_return = meta_train["return_scaled"].to_numpy(dtype=np.float32)
y_train_alpha = meta_train["alpha_scaled"].to_numpy(dtype=np.float32)
y_train_vol = meta_train["vol_scaled"].to_numpy(dtype=np.float32)
y_train_tail = meta_train["tail_scaled"].to_numpy(dtype=np.float32)
y_train_downside = meta_train["target_bottom_quintile"].to_numpy(dtype=np.float32)
y_train_regime = meta_train["market_regime_code"].to_numpy(dtype=np.int64)
w_train = meta_train["sample_weight"].to_numpy(dtype=np.float32)

y_val_return = meta_val["return_scaled"].to_numpy(dtype=np.float32)
y_val_alpha = meta_val["alpha_scaled"].to_numpy(dtype=np.float32)
y_val_vol = meta_val["vol_scaled"].to_numpy(dtype=np.float32)
y_val_tail = meta_val["tail_scaled"].to_numpy(dtype=np.float32)
y_val_downside = meta_val["target_bottom_quintile"].to_numpy(dtype=np.float32)
y_val_regime = meta_val["market_regime_code"].to_numpy(dtype=np.int64)
w_val = meta_val["sample_weight"].to_numpy(dtype=np.float32)

print("All sequences:", X_seq.shape)
print("Train sequences:", X_train_seq.shape)
print("Val sequences:", X_val_seq.shape)
print("Test sequences:", X_test_seq_all.shape)

All sequences: (72657, 20, 71)
Train sequences: (32295, 20, 71)
Val sequences: (12625, 20, 71)
Test sequences: (24942, 20, 71)


## Volatility-Focused LSTM-AE

In [8]:
class MultiRegimeRouterLSTMAutoencoder(nn.Module):
    def __init__(self, input_dim: int, hidden_size: int, latent_dim: int, num_layers: int, dropout: float, regime_classes: int):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_size = int(hidden_size)
        self.latent_dim = int(latent_dim)
        self.num_layers = int(num_layers)
        self.dropout = float(dropout)
        self.regime_classes = int(regime_classes)
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.to_latent = nn.Sequential(nn.Linear(hidden_size, latent_dim), nn.LayerNorm(latent_dim), nn.Tanh())
        self.decoder_seed = nn.Linear(latent_dim, hidden_size)
        self.decoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.reconstruction_head = nn.Linear(hidden_size, input_dim)
        self.return_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.alpha_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.vol_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.tail_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.downside_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.regime_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, regime_classes))

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        _, (h_n, _) = self.encoder(x)
        latent = self.to_latent(h_n[-1])
        repeated = self.decoder_seed(latent).unsqueeze(1).repeat(1, x.shape[1], 1)
        decoded, _ = self.decoder(repeated)
        return {
            "reconstruction": self.reconstruction_head(decoded),
            "expected_return": self.return_head(latent).squeeze(-1),
            "expected_alpha": self.alpha_head(latent).squeeze(-1),
            "expected_vol_log": self.vol_head(latent).squeeze(-1),
            "expected_tail_log": self.tail_head(latent).squeeze(-1),
            "downside_logit": self.downside_head(latent).squeeze(-1),
            "regime_logits": self.regime_head(latent),
            "latent": latent,
        }


def make_reconstruction_feature_weights(feature_names: list[str]) -> pd.Series:
    weights = pd.Series(1.0, index=feature_names, dtype=float)
    emphasis_terms = ["vol", "downside", "drawdown", "corr", "stress", "liquidity", "gap", "range", "beta", "breadth"]
    for name in feature_names:
        if any(term in name for term in emphasis_terms):
            weights.loc[name] = 2.0
    return weights / weights.mean()


reconstruction_feature_weights = make_reconstruction_feature_weights(ALL_FEATURE_NAMES)
reconstruction_weight_tensor = torch.as_tensor(reconstruction_feature_weights.to_numpy(dtype=np.float32)).view(1, 1, -1)


def apply_feature_denoising(x: torch.Tensor, mask_rate: float) -> torch.Tensor:
    if mask_rate <= 0.0:
        return x
    keep = (torch.rand_like(x) > mask_rate).to(x.dtype)
    return x * keep


def weighted_mse(pred: torch.Tensor, target: torch.Tensor, sample_weight: torch.Tensor) -> torch.Tensor:
    return (sample_weight * (pred - target).pow(2)).mean()


def pairwise_rank_loss(pred_alpha: torch.Tensor, true_alpha: torch.Tensor, date_codes: torch.Tensor) -> torch.Tensor:
    losses = []
    for date_code in torch.unique(date_codes):
        idx = torch.where(date_codes == date_code)[0]
        if idx.numel() < 2:
            continue
        pred = pred_alpha[idx]
        target = true_alpha[idx]
        target_diff = target[:, None] - target[None, :]
        useful_pairs = target_diff > 0
        if useful_pairs.any():
            pred_diff = pred[:, None] - pred[None, :]
            losses.append(F.softplus(-pred_diff[useful_pairs]).mean())
    if not losses:
        return pred_alpha.new_tensor(0.0)
    return torch.stack(losses).mean()


def batch_loss(
    model: nn.Module,
    xb: torch.Tensor,
    y_return: torch.Tensor,
    y_alpha: torch.Tensor,
    y_vol: torch.Tensor,
    y_tail: torch.Tensor,
    y_downside: torch.Tensor,
    y_regime: torch.Tensor,
    sample_weight: torch.Tensor,
    date_codes: torch.Tensor,
    *,
    denoise: bool,
) -> tuple[torch.Tensor, dict[str, float]]:
    clean_x = xb
    noisy_x = apply_feature_denoising(clean_x, MASK_RATE) if denoise else clean_x
    output = model(noisy_x)
    feature_weights = reconstruction_weight_tensor.to(clean_x.device)
    recon_loss = ((output["reconstruction"] - clean_x).pow(2) * feature_weights).mean()
    return_loss = weighted_mse(output["expected_return"], y_return, sample_weight)
    alpha_loss = weighted_mse(output["expected_alpha"], y_alpha, sample_weight)
    vol_loss = weighted_mse(output["expected_vol_log"], y_vol, sample_weight)
    tail_loss = weighted_mse(output["expected_tail_log"], y_tail, sample_weight)
    downside_loss = F.binary_cross_entropy_with_logits(output["downside_logit"], y_downside, weight=sample_weight)
    regime_loss = F.cross_entropy(output["regime_logits"], y_regime, reduction="none")
    regime_loss = (regime_loss * sample_weight).mean()
    rank_loss = pairwise_rank_loss(output["expected_alpha"], y_alpha, date_codes)
    total = (
        RECON_WEIGHT * recon_loss
        + RETURN_WEIGHT * return_loss
        + ALPHA_WEIGHT * alpha_loss
        + VOL_WEIGHT * vol_loss
        + TAIL_WEIGHT * tail_loss
        + DOWNSIDE_WEIGHT * downside_loss
        + REGIME_WEIGHT * regime_loss
        + RANK_WEIGHT * rank_loss
    )
    parts = {
        "loss": float(total.detach().cpu()),
        "recon": float(recon_loss.detach().cpu()),
        "return": float(return_loss.detach().cpu()),
        "alpha": float(alpha_loss.detach().cpu()),
        "vol": float(vol_loss.detach().cpu()),
        "tail": float(tail_loss.detach().cpu()),
        "downside": float(downside_loss.detach().cpu()),
        "regime": float(regime_loss.detach().cpu()),
        "rank": float(rank_loss.detach().cpu()),
    }
    return total, parts


def iter_date_batches(X, y_return, y_alpha, y_vol, y_tail, y_downside, y_regime, sample_weight, dates, *, dates_per_batch, shuffle, rng):
    date_series = pd.Series(pd.to_datetime(dates))
    unique_dates = np.array(pd.unique(date_series))
    if shuffle:
        rng.shuffle(unique_dates)
    for start in range(0, len(unique_dates), dates_per_batch):
        chosen_dates = unique_dates[start : start + dates_per_batch]
        idx = np.flatnonzero(np.isin(date_series.to_numpy(), chosen_dates))
        if shuffle:
            rng.shuffle(idx)
        date_codes = pd.factorize(date_series.iloc[idx])[0].astype(np.int64)
        yield X[idx], y_return[idx], y_alpha[idx], y_vol[idx], y_tail[idx], y_downside[idx], y_regime[idx], sample_weight[idx], date_codes

In [9]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = MultiRegimeRouterLSTMAutoencoder(
    input_dim=len(ALL_FEATURE_NAMES),
    hidden_size=HIDDEN_SIZE,
    latent_dim=LATENT_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    regime_classes=len(REGIME_CLASSES),
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
train_dates = meta_train["date"].to_numpy(dtype="datetime64[ns]")
val_dates = meta_val["date"].to_numpy(dtype="datetime64[ns]")


def evaluate_losses(model: nn.Module) -> dict[str, float]:
    model.eval()
    rng = np.random.default_rng(RANDOM_SEED)
    keys = ["loss", "recon", "return", "alpha", "vol", "tail", "downside", "regime", "rank"]
    totals = {key: 0.0 for key in keys}
    totals["n"] = 0.0
    with torch.no_grad():
        for xb_np, yr_np, ya_np, yv_np, yt_np, yd_np, yg_np, sw_np, dc_np in iter_date_batches(
            X_val_seq, y_val_return, y_val_alpha, y_val_vol, y_val_tail, y_val_downside, y_val_regime, w_val, val_dates,
            dates_per_batch=DATES_PER_BATCH, shuffle=False, rng=rng,
        ):
            xb = torch.as_tensor(xb_np, dtype=torch.float32, device=device)
            yr = torch.as_tensor(yr_np, dtype=torch.float32, device=device)
            ya = torch.as_tensor(ya_np, dtype=torch.float32, device=device)
            yv = torch.as_tensor(yv_np, dtype=torch.float32, device=device)
            yt = torch.as_tensor(yt_np, dtype=torch.float32, device=device)
            yd = torch.as_tensor(yd_np, dtype=torch.float32, device=device)
            yg = torch.as_tensor(yg_np, dtype=torch.long, device=device)
            sw = torch.as_tensor(sw_np, dtype=torch.float32, device=device)
            dc = torch.as_tensor(dc_np, dtype=torch.long, device=device)
            _, parts = batch_loss(model, xb, yr, ya, yv, yt, yd, yg, sw, dc, denoise=False)
            batch_n = float(len(xb_np))
            for key in keys:
                totals[key] += parts[key] * batch_n
            totals["n"] += batch_n
    return {key: totals[key] / max(totals["n"], 1.0) for key in keys}


history = []
best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
best_val_loss = float("inf")
patience_counter = 0
rng = np.random.default_rng(RANDOM_SEED)
keys = ["loss", "recon", "return", "alpha", "vol", "tail", "downside", "regime", "rank"]

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_totals = {key: 0.0 for key in keys}
    train_totals["n"] = 0.0
    for xb_np, yr_np, ya_np, yv_np, yt_np, yd_np, yg_np, sw_np, dc_np in iter_date_batches(
        X_train_seq, y_train_return, y_train_alpha, y_train_vol, y_train_tail, y_train_downside, y_train_regime, w_train, train_dates,
        dates_per_batch=DATES_PER_BATCH, shuffle=True, rng=rng,
    ):
        xb = torch.as_tensor(xb_np, dtype=torch.float32, device=device)
        yr = torch.as_tensor(yr_np, dtype=torch.float32, device=device)
        ya = torch.as_tensor(ya_np, dtype=torch.float32, device=device)
        yv = torch.as_tensor(yv_np, dtype=torch.float32, device=device)
        yt = torch.as_tensor(yt_np, dtype=torch.float32, device=device)
        yd = torch.as_tensor(yd_np, dtype=torch.float32, device=device)
        yg = torch.as_tensor(yg_np, dtype=torch.long, device=device)
        sw = torch.as_tensor(sw_np, dtype=torch.float32, device=device)
        dc = torch.as_tensor(dc_np, dtype=torch.long, device=device)

        optimizer.zero_grad(set_to_none=True)
        loss, parts = batch_loss(model, xb, yr, ya, yv, yt, yd, yg, sw, dc, denoise=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        batch_n = float(len(xb_np))
        for key in keys:
            train_totals[key] += parts[key] * batch_n
        train_totals["n"] += batch_n

    train_metrics = {key: train_totals[key] / max(train_totals["n"], 1.0) for key in keys}
    val_metrics = evaluate_losses(model)
    row = {"epoch": epoch}
    row.update({f"train_{k}": v for k, v in train_metrics.items()})
    row.update({f"val_{k}": v for k, v in val_metrics.items()})
    history.append(row)
    print(
        f"epoch {epoch:02d}/{EPOCHS} | train {train_metrics['loss']:.4f} | val {val_metrics['loss']:.4f} | "
        f"vol {val_metrics['vol']:.4f} | tail {val_metrics['tail']:.4f} | downside {val_metrics['downside']:.4f}"
    )

    if val_metrics["loss"] < best_val_loss - 1e-5:
        best_val_loss = val_metrics["loss"]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

model.load_state_dict(best_state)
history = pd.DataFrame(history)
display(history.tail())
print("Best validation loss:", best_val_loss)

epoch 01/18 | train 6.8509 | val 10.0950 | vol 2.2719 | tail 2.5475 | downside 0.8440
epoch 02/18 | train 6.1301 | val 9.9237 | vol 2.3243 | tail 2.5253 | downside 0.8444
epoch 03/18 | train 5.8219 | val 9.9867 | vol 2.2930 | tail 2.6314 | downside 0.8478
epoch 04/18 | train 5.5693 | val 10.3066 | vol 2.4271 | tail 2.7297 | downside 0.8501
epoch 05/18 | train 5.2875 | val 10.5936 | vol 2.3383 | tail 2.8822 | downside 0.8700
epoch 06/18 | train 5.0464 | val 10.7962 | vol 2.5156 | tail 2.8257 | downside 0.8771
epoch 07/18 | train 4.8222 | val 10.9516 | vol 2.5269 | tail 2.9328 | downside 0.8904
Early stopping at epoch 7.


,epoch,train_loss,train_recon,train_return,train_alpha,train_vol,train_tail,train_downside,train_regime,train_rank,val_loss,val_recon,val_return,val_alpha,val_vol,val_tail,val_downside,val_regime,val_rank
2,3,5.821924,0.586122,1.657227,1.652097,1.098975,1.356413,0.838500,0.468153,0.686244,9.986665,1.942970,2.563646,2.231306,2.293049,2.631415,0.847775,0.613061,0.699437
3,4,5.569294,0.567751,1.552994,1.565296,1.049441,1.291098,0.833979,0.416570,0.677691,10.306587,1.918014,2.650144,2.359863,2.427144,2.729703,0.850120,0.589141,0.710402
4,5,5.287486,0.553364,1.428535,1.454252,1.004705,1.214182,0.823550,0.387856,0.666486,10.593588,1.883476,2.909215,2.553595,2.338327,2.882177,0.870000,0.582040,0.724987
5,6,5.046377,0.547789,1.333600,1.349075,0.954979,1.160522,0.814618,0.376249,0.655870,10.796173,1.876734,2.883753,2.633244,2.515643,2.825718,0.877125,0.579113,0.728847
6,7,4.822237,0.539694,1.237086,1.260189,0.919186,1.093020,0.800867,0.370654,0.646233,10.951572,1.843441,3.013779,2.693958,2.526912,2.932772,0.890406,0.570382,0.742303


Best validation loss: 9.923716749059091


## Prediction Scores And Regime Diagnostics

In [10]:
def predict_scaled(model: nn.Module, X: np.ndarray, batch_size: int = 4096) -> pd.DataFrame:
    model.eval()
    returns, alphas, vols, tails, downside_probs, latent_norms, regime_probs = [], [], [], [], [], [], []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.as_tensor(X[start : start + batch_size], dtype=torch.float32, device=device)
            output = model(xb)
            returns.append(output["expected_return"].detach().cpu().numpy())
            alphas.append(output["expected_alpha"].detach().cpu().numpy())
            vols.append(output["expected_vol_log"].detach().cpu().numpy())
            tails.append(output["expected_tail_log"].detach().cpu().numpy())
            downside_probs.append(torch.sigmoid(output["downside_logit"]).detach().cpu().numpy())
            regime_probs.append(torch.softmax(output["regime_logits"], dim=1).detach().cpu().numpy())
            latent_norms.append(torch.linalg.norm(output["latent"], dim=1).detach().cpu().numpy())
    result = pd.DataFrame(
        {
            "return_scaled_pred": np.concatenate(returns),
            "alpha_scaled_pred": np.concatenate(alphas),
            "vol_scaled_pred": np.concatenate(vols),
            "tail_scaled_pred": np.concatenate(tails),
            "expected_downside_risk": np.concatenate(downside_probs),
            "latent_norm": np.concatenate(latent_norms),
        }
    )
    regime_array = np.concatenate(regime_probs)
    for idx, name in enumerate(REGIME_CLASSES):
        result[f"regime_prob_{name}"] = regime_array[:, idx]
    return result


def unscale_predictions(predicted: pd.DataFrame) -> pd.DataFrame:
    result = predicted.copy()
    result["expected_return"] = result["return_scaled_pred"] * target_stds["return"] + target_means["return"]
    result["expected_alpha"] = result["alpha_scaled_pred"] * target_stds["alpha"] + target_means["alpha"]
    vol_log = result["vol_scaled_pred"] * target_stds["vol_log"] + target_means["vol_log"]
    tail_log = result["tail_scaled_pred"] * target_stds["tail_log"] + target_means["tail_log"]
    result["expected_forward_volatility"] = np.expm1(vol_log).clip(lower=1e-4)
    result["expected_tail_loss"] = np.expm1(tail_log).clip(lower=0.0)
    result["regime_confidence"] = (
        result["regime_prob_calm_risk_on"]
        + 0.50 * result["regime_prob_rebound"]
        - 0.75 * result["regime_prob_high_vol"]
        - result["regime_prob_drawdown"]
    )
    return result


def rank_normalize_by_date(frame: pd.DataFrame, score_col: str, out_col: str) -> pd.DataFrame:
    result = frame.copy()
    pct_rank = result.groupby("date")[score_col].rank(method="average", pct=True)
    result[out_col] = 2.0 * pct_rank - 1.0
    return result


def add_scores(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["expected_volatility"] = np.clip(result["historical_vol_20d"].to_numpy(dtype=float), 1e-4, None)
    result["uncertainty"] = np.abs(result["expected_return"] - result["expected_alpha"])
    benchmark_beta = result.get("beta_60d_benchmark", result.get("beta_60d_spy", pd.Series(1.0, index=result.index)))
    result["benchmark_beta_excess"] = np.maximum(pd.to_numeric(benchmark_beta, errors="coerce").fillna(1.0) - BETA_TARGET, 0.0)
    result["alpha_hist_vol_score"] = result["expected_alpha"] / result["expected_volatility"].clip(lower=1e-4)
    result["alpha_pred_vol_score"] = result["expected_alpha"] / result["expected_forward_volatility"].clip(lower=1e-4)
    result["vol_blended_score"] = (
        result["expected_alpha"]
        / (0.50 * result["expected_volatility"] + 0.50 * result["expected_forward_volatility"] + 1e-4)
        - DOWNSIDE_SCORE_PENALTY * result["expected_downside_risk"]
        - TAIL_SCORE_PENALTY * result["expected_tail_loss"]
        - UNCERTAINTY_SCORE_PENALTY * result["uncertainty"].abs()
        + REGIME_SCORE_BONUS * result["regime_confidence"]
    )
    result["strong_downside_score"] = (
        result["expected_alpha"]
        / (0.25 * result["expected_volatility"] + 0.75 * result["expected_forward_volatility"] + 1e-4)
        - 1.35 * DOWNSIDE_SCORE_PENALTY * result["expected_downside_risk"]
        - 1.50 * TAIL_SCORE_PENALTY * result["expected_tail_loss"]
        - 0.50 * result["regime_prob_drawdown"]
        - 0.25 * result["regime_prob_high_vol"]
        - 0.18 * result["benchmark_beta_excess"]
    )
    rebound_gate = (
        (result["bench_momentum_20d"].fillna(0.0) > 0.0)
        & (result["rebound_breadth_confirmation"].fillna(0.0) > 0.0)
        & (result["expected_downside_risk"] < 0.35)
        & (result["expected_tail_loss"] < result.groupby("date")["expected_tail_loss"].transform("median"))
    ).astype(float)
    result["rebound_capture_score"] = (
        result["alpha_pred_vol_score"]
        + 0.20 * rebound_gate
        + 0.08 * result["regime_confidence"]
        - 0.55 * result["expected_downside_risk"]
        - 0.75 * result["expected_tail_loss"]
    )
    stress_mask = (result["regime_prob_drawdown"] > 0.30) | (result["regime_prob_high_vol"] > 0.35) | (result["bench_stress_state"].fillna(0.0) > 0)
    rebound_mask = rebound_gate > 0
    result["state_machine_score"] = result["vol_blended_score"]
    result.loc[stress_mask, "state_machine_score"] = result.loc[stress_mask, "strong_downside_score"]
    result.loc[rebound_mask, "state_machine_score"] = result.loc[rebound_mask, "rebound_capture_score"]
    result["multi_regime_router_score"] = (
        result["state_machine_score"]
        - 0.20 * result["benchmark_beta_excess"]
        - 0.10 * result["uncertainty"].abs()
        + 0.08 * result["rebound_breadth_confirmation"].fillna(0.0)
    )
    for col in ["alpha_hist_vol_score", "alpha_pred_vol_score", "vol_blended_score", "strong_downside_score", "rebound_capture_score", "state_machine_score", "multi_regime_router_score"]:
        result = rank_normalize_by_date(result, col, f"{col}_rank")
    return result


def daily_rank_ic(frame: pd.DataFrame, score_col: str, target_col: str) -> pd.DataFrame:
    rows = []
    for date_value, group in frame.groupby("date", sort=True):
        if group[score_col].nunique() < 2 or group[target_col].nunique() < 2:
            continue
        rows.append({"date": pd.Timestamp(date_value), "rank_ic": group[score_col].rank().corr(group[target_col].rank())})
    return pd.DataFrame(rows)


def diagnostics_by_regime(frame: pd.DataFrame, score_col: str) -> pd.DataFrame:
    rows = []
    for regime_name, group in frame.groupby("market_regime", sort=True):
        rank_ic = daily_rank_ic(group, score_col, "target_alpha")
        rows.append(
            {
                "market_regime": regime_name,
                "rows": len(group),
                "rank_ic": float(rank_ic["rank_ic"].mean()) if not rank_ic.empty else np.nan,
                "avg_forward_vol": float(group["target_forward_vol"].mean()),
                "avg_tail_loss": float(group["target_tail_loss"].mean()),
                "avg_downside_prob": float(group["expected_downside_risk"].mean()),
            }
        )
    return pd.DataFrame(rows)


val_pred_scaled = unscale_predictions(predict_scaled(model, X_val_seq))
val_diagnostics = add_scores(pd.concat([meta_val.copy(), val_pred_scaled], axis=1))
val_return_mse = float(np.mean((val_diagnostics["expected_return"] - val_diagnostics["target_return"]) ** 2))
val_alpha_mse = float(np.mean((val_diagnostics["expected_alpha"] - val_diagnostics["target_alpha"]) ** 2))
val_vol_mse = float(np.mean((val_diagnostics["expected_forward_volatility"] - val_diagnostics["target_forward_vol"]) ** 2))
val_tail_mse = float(np.mean((val_diagnostics["expected_tail_loss"] - val_diagnostics["target_tail_loss"]) ** 2))
val_downside_brier = float(np.mean((val_diagnostics["expected_downside_risk"] - val_diagnostics["target_bottom_quintile"]) ** 2))
val_regime_accuracy = float((val_diagnostics[[f"regime_prob_{name}" for name in REGIME_CLASSES]].to_numpy().argmax(axis=1) == val_diagnostics["market_regime_code"].to_numpy()).mean())

score_diagnostic_rows = []
for score_col in ["alpha_hist_vol_score", "alpha_pred_vol_score", "vol_blended_score", "strong_downside_score", "rebound_capture_score", "state_machine_score", "multi_regime_router_score"]:
    rank_ic = daily_rank_ic(val_diagnostics, score_col, "target_alpha")
    vol_rank_ic = daily_rank_ic(val_diagnostics, score_col, "target_forward_vol")
    score_diagnostic_rows.append(
        {
            "score_col": score_col,
            "rank_ic_alpha": float(rank_ic["rank_ic"].mean()) if not rank_ic.empty else np.nan,
            "rank_ic_forward_vol": float(vol_rank_ic["rank_ic"].mean()) if not vol_rank_ic.empty else np.nan,
            "avg_selected_downside_proxy": float(val_diagnostics.groupby("date").apply(lambda g: g.nlargest(max(1, len(g)//5), score_col)["target_tail_loss"].mean()).mean()),
        }
    )
score_validation_table = pd.DataFrame(score_diagnostic_rows)

def calibration_by_quantile(frame: pd.DataFrame, pred_col: str, realized_col: str, label: str, bins: int = 5) -> pd.DataFrame:
    work = frame.loc[:, [pred_col, realized_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if work.empty or work[pred_col].nunique() < 2:
        return pd.DataFrame(columns=["calibration_target", "bucket", "count", "predicted_mean", "realized_mean"])
    work["bucket"] = pd.qcut(work[pred_col].rank(method="first"), q=min(bins, len(work)), labels=False, duplicates="drop")
    grouped = work.groupby("bucket", sort=True).agg(count=(pred_col, "size"), predicted_mean=(pred_col, "mean"), realized_mean=(realized_col, "mean")).reset_index()
    grouped.insert(0, "calibration_target", label)
    return grouped

calibration_table = pd.concat([
    calibration_by_quantile(val_diagnostics, "expected_forward_volatility", "target_forward_vol", "forward_volatility"),
    calibration_by_quantile(val_diagnostics, "expected_tail_loss", "target_tail_loss", "tail_loss"),
    calibration_by_quantile(val_diagnostics, "expected_downside_risk", "target_bottom_quintile", "bottom_quintile_probability"),
    calibration_by_quantile(val_diagnostics, "uncertainty", "target_tail_loss", "uncertainty_vs_tail_loss"),
], ignore_index=True)

print("Validation MSE/Brier:", {"return": val_return_mse, "alpha": val_alpha_mse, "vol": val_vol_mse, "tail": val_tail_mse, "downside": val_downside_brier, "regime_acc": val_regime_accuracy})
display(score_validation_table)
print("Calibration table:")
display(calibration_table)
print("Regime diagnostics for strong downside score:")
display(diagnostics_by_regime(val_diagnostics, "strong_downside_score"))

Validation MSE/Brier: {'return': 0.0031553407205477725, 'alpha': 0.0019926491964248117, 'vol': 0.05563154598429339, 'tail': 0.0005715734252782068, 'downside': 0.15973182269364505, 'regime_acc': 0.7771881188118812}


,score_col,rank_ic_alpha,rank_ic_forward_vol,avg_selected_downside_proxy
0,alpha_hist_vol_score,0.031042,-0.169704,0.022496
1,alpha_pred_vol_score,0.047768,-0.036445,0.025212
2,vol_blended_score,0.014547,-0.321107,0.020870
3,strong_downside_score,0.032885,-0.389717,0.020104
4,rebound_capture_score,0.026704,-0.273162,0.021464
5,state_machine_score,0.030069,-0.389549,0.020231
6,multi_regime_router_score,0.032219,-0.379455,0.020567


Calibration table:


,calibration_target,bucket,count,predicted_mean,realized_mean
0,forward_volatility,0,2525,0.171612,0.187025
1,forward_volatility,1,2525,0.212898,0.231649
2,forward_volatility,2,2525,0.241354,0.293952
3,forward_volatility,3,2525,0.268960,0.353847
4,forward_volatility,4,2525,0.327340,0.538273
5,tail_loss,0,2525,0.014883,0.015027
6,tail_loss,1,2525,0.017736,0.019720
7,tail_loss,2,2525,0.019751,0.023614
8,tail_loss,3,2525,0.021948,0.030361
9,tail_loss,4,2525,0.027167,0.039839


Regime diagnostics for strong downside score:


,market_regime,rows,rank_ic,avg_forward_vol,avg_tail_loss,avg_downside_prob
0,drawdown,3850,-0.014141,0.430204,0.032909,0.143622
1,high_vol,7800,0.059842,0.260337,0.021647,0.153451
2,rebound,975,0.002919,0.374430,0.029818,0.197559


## Multi-Regime Router Portfolio Construction

In [11]:
def select_rebalance_dates(dates: pd.Series | pd.DatetimeIndex, frequency: str = "weekly") -> pd.DatetimeIndex:
    unique_dates = pd.DatetimeIndex(pd.to_datetime(pd.Series(dates).drop_duplicates()).sort_values())
    if frequency == "daily":
        return unique_dates
    date_series = pd.Series(unique_dates, index=unique_dates)
    if frequency == "weekly":
        return pd.DatetimeIndex(date_series.groupby(date_series.index.to_period("W-FRI")).max().to_numpy())
    if frequency == "monthly":
        return pd.DatetimeIndex(date_series.groupby(date_series.index.to_period("M")).max().to_numpy())
    if frequency == "every_5_trading_days":
        return unique_dates[::5]
    if frequency == "every_10_trading_days":
        return unique_dates[::10]
    raise ValueError("frequency must be one of: daily, weekly, monthly, every_5_trading_days, every_10_trading_days")


def apply_weight_cap(raw_weights: pd.Series, max_weight: float) -> pd.Series:
    raw = raw_weights.clip(lower=0.0).astype(float)
    raw = raw.loc[raw > 0.0]
    if raw.empty:
        return raw_weights * 0.0
    feasible_cap = max(float(max_weight), 1.0 / len(raw) + 1e-8)
    weights = pd.Series(0.0, index=raw.index, dtype=float)
    remaining = raw.copy()
    budget = 1.0
    while not remaining.empty:
        scaled = remaining / remaining.sum() * budget
        over_cap = scaled > feasible_cap
        if not over_cap.any():
            weights.loc[remaining.index] = scaled
            break
        capped_names = scaled.loc[over_cap].index
        weights.loc[capped_names] = feasible_cap
        budget -= feasible_cap * len(capped_names)
        remaining = remaining.drop(index=capped_names)
        if budget <= 1e-12:
            break
    total = weights.sum()
    if total <= 0.0:
        weights = pd.Series(1.0 / len(raw), index=raw.index, dtype=float)
    elif abs(total - 1.0) > 1e-10:
        weights = weights / total
    return weights


def _price_wide(prices: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    wide = prices.copy()
    wide["date"] = pd.to_datetime(wide["date"], utc=True).dt.tz_localize(None)
    wide["ticker"] = wide["ticker"].astype(str).str.upper()
    return wide.pivot(index="date", columns="ticker", values="adj_close").sort_index().reindex(columns=tickers)


price_wide = _price_wide(prices, tradable_tickers)
returns_wide = price_wide.pct_change()


def effective_name_count(weights: pd.Series) -> float:
    values = weights.fillna(0.0).to_numpy(dtype=float)
    denom = float(np.square(values).sum())
    return 1.0 / denom if denom > 0 else 0.0


def min_variance_anchor(date_value: pd.Timestamp, tickers: list[str], fallback_vol: pd.Series, max_weight: float) -> pd.Series:
    available_tickers = [ticker for ticker in tickers if ticker in returns_wide.columns]
    if not available_tickers:
        return pd.Series(1.0 / len(tickers), index=tickers, dtype=float)
    hist = returns_wide.loc[:pd.Timestamp(date_value), available_tickers].tail(COV_LOOKBACK_DAYS).dropna(how="all")
    if len(hist) < 40 or len(available_tickers) < 2:
        inv = 1.0 / fallback_vol.reindex(tickers).replace(0.0, np.nan).fillna(fallback_vol.median()).clip(lower=1e-4)
        return apply_weight_cap(inv, max_weight=max_weight).reindex(tickers).fillna(0.0)
    clean = hist.fillna(0.0)
    try:
        cov = LedoitWolf().fit(clean.to_numpy(dtype=float)).covariance_
        inv_cov = np.linalg.pinv(cov + np.eye(cov.shape[0]) * 1e-8)
        ones = np.ones(len(available_tickers))
        raw = inv_cov @ ones
        raw = np.clip(raw, 0.0, None)
        if raw.sum() <= 0.0:
            raw = 1.0 / np.diag(cov).clip(min=1e-8)
        anchor = pd.Series(raw, index=available_tickers, dtype=float)
    except Exception:
        inv = 1.0 / fallback_vol.reindex(available_tickers).replace(0.0, np.nan).fillna(fallback_vol.median()).clip(lower=1e-4)
        anchor = inv.astype(float)
    anchor = apply_weight_cap(anchor, max_weight=max_weight)
    return anchor.reindex(tickers).fillna(0.0)


def _market_state(group: pd.DataFrame, proxy_drawdown: float) -> str:
    bench_mom = float(group.get("bench_momentum_20d", pd.Series(0.0, index=group.index)).median())
    spy_mom = float(group.get("spy_momentum_20d", pd.Series(0.0, index=group.index)).median())
    breadth_confirm = float(group.get("rebound_breadth_confirmation", pd.Series(0.0, index=group.index)).mean())
    bench_stress = float(group.get("bench_stress_state", pd.Series(0.0, index=group.index)).mean())
    stress = (
        float(group["spy_vol_20d_ann"].median()) >= regime_thresholds["high_vol_spy_vol_20d_ann"]
        or float(group.get("bench_vol_20d_ann", group["spy_vol_20d_ann"]).median()) >= regime_thresholds.get("bench_high_vol_20d_ann", regime_thresholds["high_vol_spy_vol_20d_ann"])
        or float(group["spy_drawdown_60d"].median()) <= regime_thresholds["drawdown_spy_drawdown_60d"]
        or float(group.get("bench_drawdown_60d", group["spy_drawdown_60d"]).median()) <= regime_thresholds.get("bench_drawdown_60d", regime_thresholds["drawdown_spy_drawdown_60d"])
        or float(group["benchmark_agnostic_stress"].median()) >= regime_thresholds["stress_benchmark_agnostic"]
        or float(group["regime_prob_drawdown"].mean()) >= 0.28
        or float(group["regime_prob_high_vol"].mean()) >= 0.35
        or bench_stress > 0.50
        or proxy_drawdown <= -0.12
    )
    rebound = (bench_mom > 0.0 or spy_mom > 0.0) and breadth_confirm > 0.20 and float(group["expected_downside_risk"].mean()) < 0.38
    risk_on = (bench_mom > 0.03 and spy_mom > 0.01 and float(group["regime_confidence"].mean()) > 0.0 and float(group["expected_tail_loss"].mean()) < group["expected_tail_loss"].median())
    if stress and rebound:
        return "rebound"
    if stress:
        return "stress"
    if risk_on:
        return "risk_on"
    if rebound:
        return "rebound"
    return "neutral"


def weights_from_multiregime_router_scores(
    predictions: pd.DataFrame,
    *,
    score_col: str,
    dataset_name,
    strategy_name: str,
    frequency: str,
    high_vol_max_weight: float,
    normal_max_weight: float,
    turnover_blend: float,
    beta_target: float,
    universe_tickers: list[str],
) -> tuple[PortfolioWeights, pd.DataFrame]:
    tickers = [ticker for ticker in universe_tickers if ticker in set(predictions["ticker"])]
    rows = []
    logs = []
    previous = None
    previous_date = None
    proxy_nav = 1.0
    proxy_peak = 1.0

    for date_value, group in predictions.groupby("date", sort=True):
        date_value = pd.Timestamp(date_value)
        if previous is not None and previous_date is not None and previous_date in price_wide.index and date_value in price_wide.index:
            period_returns = price_wide.loc[date_value, tickers] / price_wide.loc[previous_date, tickers] - 1.0
            period_returns = period_returns.replace([np.inf, -np.inf], np.nan).fillna(0.0)
            proxy_nav *= float(1.0 + (previous * period_returns).sum())
            proxy_peak = max(proxy_peak, proxy_nav)
        proxy_drawdown = proxy_nav / max(proxy_peak, 1e-12) - 1.0

        market_state = _market_state(group, proxy_drawdown)
        unfavorable = market_state == "stress"
        if market_state == "stress":
            effective_max_weight = high_vol_max_weight
        elif market_state == "rebound":
            effective_max_weight = min(normal_max_weight, max(high_vol_max_weight, 0.075))
        elif market_state == "risk_on":
            effective_max_weight = normal_max_weight
        else:
            effective_max_weight = min(normal_max_weight, max(high_vol_max_weight, 0.075))
        if proxy_drawdown <= -0.08:
            effective_max_weight = min(effective_max_weight, 0.05)
        if proxy_drawdown <= -0.15:
            effective_max_weight = min(effective_max_weight, 0.03)
        if market_state == "stress":
            min_names = HIGH_RISK_MIN_ACTIVE_NAMES
            target_fraction = 0.90
        elif market_state == "rebound":
            min_names = max(NORMAL_MIN_ACTIVE_NAMES, int(0.45 * len(tickers)))
            target_fraction = 0.55
        elif market_state == "risk_on":
            min_names = NORMAL_MIN_ACTIVE_NAMES
            target_fraction = 0.45
        else:
            min_names = NORMAL_MIN_ACTIVE_NAMES
            target_fraction = BASE_TOP_FRACTION
        min_names = min(len(tickers), max(min_names, int(math.ceil(1.0 / effective_max_weight))))

        group = group.copy()
        tail_cutoff = group["expected_tail_loss"].quantile(0.85)
        downside_cutoff = group["expected_downside_risk"].quantile(0.85)
        quality = group.loc[(group["expected_tail_loss"] <= tail_cutoff) & (group["expected_downside_risk"] <= downside_cutoff)].copy()
        if len(quality) >= min_names:
            group = quality
        group = group.sort_values(score_col, ascending=False).reset_index(drop=True)
        top_n = min(len(group), max(min_names, int(math.ceil(len(group) * target_fraction))))
        selected = group.head(top_n).copy()
        selected_tickers = [ticker for ticker in selected["ticker"] if ticker in tickers]
        fallback_vol = selected.set_index("ticker")["expected_forward_volatility"].reindex(selected_tickers).fillna(selected["expected_forward_volatility"].median())
        anchor = min_variance_anchor(date_value, selected_tickers, fallback_vol, max_weight=effective_max_weight)

        raw_signal = selected.set_index("ticker")[score_col].reindex(selected_tickers).astype(float)
        if score_col.endswith("_rank"):
            tilt_raw = (raw_signal - raw_signal.min() + 1e-6)
        else:
            tilt_raw = raw_signal.rank(pct=True).clip(lower=0.0)
        risk_divisor = (
            selected.set_index("ticker")["expected_forward_volatility"].reindex(selected_tickers).clip(lower=1e-4)
            * (1.0 + selected.set_index("ticker")["expected_downside_risk"].reindex(selected_tickers).fillna(0.5))
            * (1.0 + 10.0 * selected.set_index("ticker")["expected_tail_loss"].reindex(selected_tickers).fillna(0.0))
        )
        tilt_raw = (tilt_raw / risk_divisor).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        tilt = apply_weight_cap(tilt_raw, max_weight=effective_max_weight).reindex(selected_tickers).fillna(0.0)
        confidence_ok = (
            float(selected["expected_downside_risk"].mean()) < 0.35
            and float(selected["regime_confidence"].mean()) > -0.15
            and float(selected["uncertainty"].mean()) < selected["uncertainty"].quantile(0.75)
        )
        if market_state == "stress":
            tilt_strength = MINVAR_TILT_HIGH_RISK
        elif market_state == "rebound":
            tilt_strength = min(0.38, MINVAR_TILT_NORMAL * 1.15)
        elif market_state == "risk_on":
            tilt_strength = min(0.45, MINVAR_TILT_NORMAL * 1.25)
        else:
            tilt_strength = MINVAR_TILT_NORMAL
        if not confidence_ok:
            tilt_strength *= 0.50
        if proxy_drawdown <= -0.08:
            tilt_strength *= 0.50

        combined = (1.0 - tilt_strength) * anchor + tilt_strength * tilt
        combined = apply_weight_cap(combined, max_weight=effective_max_weight)
        row = pd.Series(0.0, index=tickers, dtype=float)
        row.loc[combined.index] = combined.to_numpy(dtype=float)
        row = row / row.sum()

        beta_source = group.drop_duplicates("ticker").set_index("ticker").reindex(tickers)
        betas = pd.to_numeric(beta_source.get("beta_60d_benchmark"), errors="coerce")
        betas = betas.fillna(pd.to_numeric(beta_source.get("beta_60d_spy"), errors="coerce")).fillna(1.0)
        portfolio_beta = float((row * betas).sum())
        beta_blend = 0.0
        if portfolio_beta > beta_target:
            inv_vol = 1.0 / group.drop_duplicates("ticker").set_index("ticker").reindex(tickers)["expected_forward_volatility"].astype(float).clip(lower=1e-4)
            inv_vol = apply_weight_cap(inv_vol.fillna(inv_vol.median()), max_weight=effective_max_weight).reindex(tickers).fillna(0.0)
            inv_beta = float((inv_vol * betas).sum())
            if portfolio_beta > inv_beta:
                beta_blend = float(np.clip((portfolio_beta - beta_target) / max(portfolio_beta - inv_beta, 1e-6), 0.0, 0.85))
                row = (1.0 - beta_blend) * row + beta_blend * inv_vol
                row = row / row.sum()
                portfolio_beta = float((row * betas).sum())

        pre_turnover_row = row.copy()
        if previous is not None:
            row = turnover_blend * row + (1.0 - turnover_blend) * previous
            row = row / row.sum()
        turnover_estimate = float((row.fillna(0.0) - (previous.fillna(0.0) if previous is not None else 0.0)).abs().sum() / 2.0)
        row.name = date_value
        rows.append(row)
        logs.append(
            {
                "date": date_value,
                "score_col": score_col,
                "frequency": frequency,
                "market_state": market_state,
                "unfavorable_regime": unfavorable,
                "effective_max_weight": effective_max_weight,
                "selected_names": len(selected_tickers),
                "active_names": int((row > 1e-8).sum()),
                "effective_names": effective_name_count(row),
                "realized_max_weight": float(row.max()),
                "portfolio_beta_estimate": portfolio_beta,
                "beta_blend": beta_blend,
                "beta_target_gap": max(0.0, portfolio_beta - beta_target),
                "low_confidence_fallback": not confidence_ok,
                "tilt_strength": tilt_strength,
                "turnover_estimate": turnover_estimate,
                "proxy_drawdown": proxy_drawdown,
            }
        )
        previous = row
        previous_date = date_value

    weights = pd.DataFrame(rows)
    weights.index.name = "date"
    metadata = {
        "score_col": score_col,
        "rebalance_frequency": frequency,
        "normal_max_weight": normal_max_weight,
        "high_vol_max_weight": high_vol_max_weight,
        "turnover_blend": turnover_blend,
        "beta_target": beta_target,
        "minvar_anchor": True,
        "universe_ticker_count": len(tickers),
    }
    return PortfolioWeights(weights=weights, dataset_name=dataset_name.identifier, strategy_name=strategy_name, metadata=metadata), pd.DataFrame(logs)

In [12]:
test_pred_scaled = unscale_predictions(predict_scaled(model, X_test_seq_all))
test_scored_all = add_scores(pd.concat([meta_test_all.copy(), test_pred_scaled], axis=1))

score_columns = ["alpha_hist_vol_score", "alpha_pred_vol_score", "vol_blended_score", "strong_downside_score", "rebound_capture_score", "state_machine_score", "multi_regime_router_score"]
if FAST_SMOKE or SUBMISSION_MINIMAL_GRID:
    max_weight_grid = [0.05]
    score_grid = ["multi_regime_router_score"]
    frequency_grid = ["weekly"]
    beta_target_grid = [BETA_TARGET]
elif FULL_ROUTER_GRID:
    max_weight_grid = HIGH_VOL_MAX_WEIGHT_OPTIONS
    score_grid = score_columns
    frequency_grid = ["weekly", "every_5_trading_days", "every_10_trading_days"]
    beta_target_grid = BETA_TARGET_GRID
else:
    # Default submission grid: small enough to finish, broad enough to test risk routing.
    # Set FULL_ROUTER_GRID=1 for the exhaustive experiment grid.
    max_weight_grid = [0.03, 0.05]
    score_grid = ["strong_downside_score", "state_machine_score", "multi_regime_router_score"]
    frequency_grid = ["weekly", "every_10_trading_days"]
    beta_target_grid = [0.70, 0.85]

candidate_portfolios = {}
candidate_logs = {}
candidate_predictions = {}
for frequency in frequency_grid:
    rebalance_dates = select_rebalance_dates(test_scored_all["date"], frequency)
    scored = test_scored_all.loc[test_scored_all["date"].isin(rebalance_dates)].copy()
    for score_col in score_grid:
        for high_vol_max_weight in max_weight_grid:
            for beta_target_value in beta_target_grid:
                name = f"{frequency}_{score_col}_hvw{str(high_vol_max_weight).replace('.', '')}_beta{str(beta_target_value).replace('.', '')}"
                prediction_frame = scored.loc[:, [
                "date", "ticker", "expected_return", "expected_alpha", "expected_volatility", "expected_forward_volatility",
                "expected_tail_loss", "expected_downside_risk", "uncertainty", "benchmark_beta_excess", "regime_confidence",
                "regime_prob_calm_risk_on", "regime_prob_high_vol", "regime_prob_drawdown", "regime_prob_rebound",
                "spy_vol_20d_ann", "spy_drawdown_60d", "spy_momentum_20d", "beta_60d_spy", "benchmark_agnostic_stress",
                "bench_vol_20d_ann", "bench_drawdown_60d", "bench_momentum_20d", "bench_rebound_state", "bench_stress_state", "beta_60d_benchmark", "dual_context_vol_spread", "rebound_breadth_confirmation",
                *score_columns, *[f"{col}_rank" for col in score_columns],
            ]].copy()
                prediction_frame["horizon"] = HORIZON
                prediction_frame = prediction_frame.loc[:, ["date", "ticker", "horizon"] + [c for c in prediction_frame.columns if c not in {"date", "ticker", "horizon"}]]
                prediction_frame = validate_prediction_frame(prediction_frame, dataset_name=research_spec, horizon=HORIZON, repo_root=repo_root)
                rank_score_col = f"{score_col}_rank"
                portfolio_obj, log_frame = weights_from_multiregime_router_scores(
                prediction_frame,
                score_col=rank_score_col,
                dataset_name=research_spec,
                strategy_name=f"{MODEL_NAME}_{name}",
                frequency=frequency,
                high_vol_max_weight=high_vol_max_weight,
                normal_max_weight=NORMAL_MAX_WEIGHT,
                turnover_blend=TURNOVER_BLEND,
                beta_target=beta_target_value,
                universe_tickers=tradable_tickers,
            )
                candidate_portfolios[name] = portfolio_obj
                candidate_logs[name] = log_frame
                candidate_predictions[name] = prediction_frame

print("Candidate count:", len(candidate_portfolios))
first_name = next(iter(candidate_portfolios))
display(candidate_predictions[first_name].head())
display(candidate_logs[first_name].head())

Candidate count: 1


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,expected_forward_volatility,expected_tail_loss,expected_downside_risk,uncertainty,...,rebound_capture_score,state_machine_score,multi_regime_router_score,alpha_hist_vol_score_rank,alpha_pred_vol_score_rank,vol_blended_score_rank,strong_downside_score_rank,rebound_capture_score_rank,state_machine_score_rank,multi_regime_router_score_rank
0,2022-01-07,AAPL,5,0.008404,0.006818,0.291344,0.233316,0.017686,0.100993,0.001586,...,-0.110607,-0.562178,-0.621337,0.04,0.52,0.92,0.52,0.92,0.52,0.60
1,2022-01-07,ADBE,5,0.005455,0.003928,0.519157,0.254422,0.020204,0.124629,0.001527,...,-0.132217,-0.680802,-0.922686,-0.92,-0.92,-0.36,-0.36,-0.52,-0.36,-0.60
2,2022-01-07,ADI,5,0.007133,0.005582,0.242832,0.253624,0.019539,0.121196,0.001551,...,-0.128256,-0.587835,-0.673255,-0.04,-0.44,0.20,0.20,-0.28,0.20,0.28
3,2022-01-07,AMAT,5,0.009963,0.009855,0.408457,0.277097,0.021766,0.127297,0.000108,...,-0.120832,-0.703052,-0.897530,0.20,0.68,0.12,-0.60,0.28,-0.60,-0.44
4,2022-01-07,AMD,5,0.014073,0.016175,0.588302,0.320840,0.024107,0.166784,0.002102,...,-0.125625,-0.799054,-1.102699,0.60,0.92,-0.68,-0.76,0.04,-0.76,-0.76


,date,score_col,frequency,market_state,unfavorable_regime,effective_max_weight,selected_names,active_names,effective_names,realized_max_weight,portfolio_beta_estimate,beta_blend,beta_target_gap,low_confidence_fallback,tilt_strength,turnover_estimate,proxy_drawdown
0,2022-01-07,multi_regime_router_score_rank,weekly,stress,True,0.05,25,25,21.432318,0.05,1.385738,0.0,0.535738,True,0.050,0.500000,0.000000
1,2022-01-14,multi_regime_router_score_rank,weekly,stress,True,0.05,25,25,21.902039,0.05,1.480285,0.0,0.630285,True,0.050,0.022247,0.000000
2,2022-01-21,multi_regime_router_score_rank,weekly,stress,True,0.05,25,25,22.132953,0.05,1.441014,0.0,0.591014,True,0.025,0.039874,-0.097823
3,2022-01-28,multi_regime_router_score_rank,weekly,stress,True,0.05,25,25,22.093232,0.05,1.444995,0.0,0.594995,True,0.025,0.018705,-0.101784
4,2022-02-04,multi_regime_router_score_rank,weekly,stress,True,0.05,25,25,22.127485,0.05,1.512685,0.0,0.662685,True,0.025,0.020340,-0.084113


## Backtests, Router Selection, And Stress Tables

In [13]:
def write_artifacts_for_notebook(result, output_dir: Path) -> dict[str, str]:
    if os.environ.get("SKIP_QUANTSTATS", "0") != "1":
        return write_backtest_artifacts(result, output_dir)
    output_path = Path(output_dir).resolve()
    output_path.mkdir(parents=True, exist_ok=True)
    paths = {
        "weights": output_path / "weights.parquet",
        "nav": output_path / "nav.parquet",
        "returns": output_path / "returns.parquet",
        "turnover": output_path / "turnover.parquet",
        "benchmarks": output_path / "benchmarks.parquet",
        "metrics": output_path / "metrics.json",
        "metrics_table": output_path / "metrics_table.parquet",
        "quantstats_report": output_path / "quantstats.html",
    }
    result.weights.to_parquet(paths["weights"])
    result.nav.to_frame(name="nav").to_parquet(paths["nav"])
    result.returns.to_frame(name="returns").to_parquet(paths["returns"])
    result.turnover.to_frame(name="turnover").to_parquet(paths["turnover"])
    result.benchmark_returns.to_parquet(paths["benchmarks"])
    paths["metrics"].write_text(json.dumps(result.metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    pd.DataFrame([{"metric": key, "value": value} for key, value in sorted(result.metrics.items())]).to_parquet(paths["metrics_table"], index=False)
    paths["quantstats_report"].write_text("<html><body><p>QuantStats skipped because SKIP_QUANTSTATS=1.</p></body></html>\n", encoding="utf-8")
    artifact_paths = {key: str(value) for key, value in paths.items()}
    result.artifact_paths.update(artifact_paths)
    return artifact_paths


def stress_table(result, benchmark_name: str) -> pd.DataFrame:
    returns = result.returns.astype(float)
    benchmark_returns = result.benchmark_returns.iloc[:, 0].astype(float)
    if benchmark_name in result.benchmark_returns.columns:
        benchmark_returns = result.benchmark_returns[benchmark_name].astype(float)
    common = pd.concat([returns.rename("strategy"), benchmark_returns.rename("benchmark")], axis=1).dropna()
    if not result.weights.empty:
        common = common.loc[(common.index >= result.weights.index.min()) & (common.index <= result.weights.index.max())]
    rolling_5d = (1.0 + common["strategy"]).rolling(5).apply(np.prod, raw=True) - 1.0
    monthly = (1.0 + common["strategy"]).resample("M").prod() - 1.0
    crash_threshold = common["benchmark"].quantile(0.10)
    rebound_threshold = common["benchmark"].quantile(0.90)
    beta = common["strategy"].cov(common["benchmark"]) / max(common["benchmark"].var(), 1e-12)
    return pd.DataFrame(
        [
            {"stress_metric": "realized_volatility", "value": float(common["strategy"].std(ddof=0) * np.sqrt(252.0))},
            {"stress_metric": "beta_to_benchmark", "value": float(beta)},
            {"stress_metric": "worst_day", "value": float(common["strategy"].min())},
            {"stress_metric": "worst_5_day_period", "value": float(rolling_5d.min())},
            {"stress_metric": "worst_month", "value": float(monthly.min())},
            {"stress_metric": "avg_return_benchmark_worst_decile_days", "value": float(common.loc[common["benchmark"] <= crash_threshold, "strategy"].mean())},
            {"stress_metric": "avg_return_benchmark_rebound_days", "value": float(common.loc[common["benchmark"] >= rebound_threshold, "strategy"].mean())},
            {"stress_metric": "benchmark_worst_decile_threshold", "value": float(crash_threshold)},
        ]
    )


def regime_return_table(result, context_frame: pd.DataFrame, benchmark_name: str) -> pd.DataFrame:
    returns = result.returns.rename("strategy_return").to_frame()
    benchmark_returns = result.benchmark_returns.iloc[:, 0].rename("benchmark_return")
    if benchmark_name in result.benchmark_returns.columns:
        benchmark_returns = result.benchmark_returns[benchmark_name].rename("benchmark_return")
    context = context_frame.loc[:, ["date", "spy_vol_20d_ann", "spy_drawdown_60d", "market_regime", "benchmark_agnostic_stress"]].drop_duplicates("date").copy()
    context["date"] = pd.to_datetime(context["date"])
    frame = returns.join(benchmark_returns, how="left").reset_index()
    frame = frame.rename(columns={frame.columns[0]: "date"})
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.merge(context, on="date", how="left").dropna(subset=["strategy_return", "benchmark_return"])
    if not result.weights.empty:
        frame = frame.loc[(frame["date"] >= result.weights.index.min()) & (frame["date"] <= result.weights.index.max())]
    crash_threshold = frame["benchmark_return"].quantile(0.10)
    rebound_threshold = frame["benchmark_return"].quantile(0.90)
    masks = {
        "benchmark_up_days": frame["benchmark_return"] > 0,
        "benchmark_down_days": frame["benchmark_return"] < 0,
        "high_benchmark_vol": frame["spy_vol_20d_ann"] >= regime_thresholds["high_vol_spy_vol_20d_ann"],
        "benchmark_worst_decile_days": frame["benchmark_return"] <= crash_threshold,
        "benchmark_rebound_days": frame["benchmark_return"] >= rebound_threshold,
        "market_drawdown_regime": frame["market_regime"].eq("drawdown"),
        "market_high_vol_regime": frame["market_regime"].eq("high_vol"),
        "benchmark_agnostic_stress": frame["benchmark_agnostic_stress"] >= regime_thresholds["stress_benchmark_agnostic"],
    }
    rows = []
    for name, mask in masks.items():
        subset = frame.loc[mask]
        rows.append(
            {
                "bucket": name,
                "observations": int(len(subset)),
                "avg_strategy_return": float(subset["strategy_return"].mean()) if len(subset) else np.nan,
                "strategy_vol_ann": float(subset["strategy_return"].std(ddof=0) * np.sqrt(252.0)) if len(subset) else np.nan,
                "avg_benchmark_return": float(subset["benchmark_return"].mean()) if len(subset) else np.nan,
                "hit_rate": float((subset["strategy_return"] > 0).mean()) if len(subset) else np.nan,
            }
        )
    return pd.DataFrame(rows)


candidate_results = {}
comparison_rows = []
for name, candidate in candidate_portfolios.items():
    print("Running backtest:", name, flush=True)
    candidate_result = backtest_weights(backtest_spec, candidate, benchmark=research_spec.benchmark_ticker, repo_root=repo_root)
    candidate_result.metrics = build_metrics(candidate_result)
    candidate_results[name] = candidate_result
    stress = stress_table(candidate_result, research_spec.benchmark_ticker)
    stress_map = dict(zip(stress["stress_metric"], stress["value"]))
    log_frame = candidate_logs[name]
    row = {"candidate": name}
    for metric in ["annual_return", "annual_volatility", "sharpe", "sortino", "max_drawdown", "average_turnover", "total_return"]:
        row[metric] = candidate_result.metrics.get(metric, np.nan)
    row.update(
        {
            "stress_worst_day": stress_map.get("worst_day"),
            "stress_worst_5_day": stress_map.get("worst_5_day_period"),
            "stress_crash_day_avg": stress_map.get("avg_return_benchmark_worst_decile_days"),
            "beta_to_benchmark": stress_map.get("beta_to_benchmark"),
            "avg_active_names": float((candidate_result.weights > 1e-8).sum(axis=1).mean()),
            "avg_effective_names": float(log_frame["effective_names"].mean()) if not log_frame.empty else np.nan,
            "avg_max_weight": float(log_frame["realized_max_weight"].mean()) if not log_frame.empty else np.nan,
            "avg_beta_estimate": float(log_frame["portfolio_beta_estimate"].mean()) if not log_frame.empty else np.nan,
            "state_counts": json.dumps(log_frame["market_state"].value_counts().to_dict() if "market_state" in log_frame else {}, sort_keys=True),
        }
    )
    comparison_rows.append(row)

backtest_comparison = pd.DataFrame(comparison_rows)
backtest_comparison["router_selection_score"] = (
    backtest_comparison["annual_volatility"]
    + 0.50 * backtest_comparison["max_drawdown"].abs()
    + 2.00 * backtest_comparison["stress_worst_day"].abs()
    + 1.50 * backtest_comparison["stress_crash_day_avg"].abs()
    + 0.05 * backtest_comparison["average_turnover"].fillna(0.0)
    + 0.25 * np.maximum(backtest_comparison["beta_to_benchmark"].fillna(BETA_TARGET) - BETA_TARGET, 0.0)
    - 0.10 * backtest_comparison["sortino"].fillna(0.0)
)
backtest_comparison = backtest_comparison.sort_values(["router_selection_score", "annual_volatility", "max_drawdown"]).reset_index(drop=True)
selected_candidate_name = str(backtest_comparison.iloc[0]["candidate"])
result = candidate_results[selected_candidate_name]
portfolio = candidate_portfolios[selected_candidate_name]
risk_log = candidate_logs[selected_candidate_name]
predictions = candidate_predictions[selected_candidate_name]
validated_weights = validate_weights_frame(portfolio.weights, dataset_name=research_spec, repo_root=repo_root)
metrics = result.metrics
artifact_paths = write_artifacts_for_notebook(result, output_dir)
final_stress_table = stress_table(result, research_spec.benchmark_ticker)
final_regime_return_table = regime_return_table(result, feature_frame, research_spec.benchmark_ticker)

backtest_comparison.to_parquet(output_dir / "router_candidate_comparison.parquet", index=False)
final_stress_table.to_parquet(output_dir / "router_stress_table.parquet", index=False)
final_regime_return_table.to_parquet(output_dir / "router_regime_return_table.parquet", index=False)
risk_log.to_parquet(output_dir / "router_risk_log.parquet", index=False)

def winner_participation_table(weights: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    rows = []
    for ticker in tickers:
        if ticker not in weights.columns:
            rows.append({"ticker": ticker, "available": False, "held_share": 0.0, "avg_weight_when_held": 0.0, "max_weight": 0.0})
            continue
        series = weights[ticker].astype(float)
        held = series > 1e-8
        rows.append({
            "ticker": ticker,
            "available": True,
            "held_share": float(held.mean()),
            "avg_weight_when_held": float(series.loc[held].mean()) if held.any() else 0.0,
            "max_weight": float(series.max()),
        })
    return pd.DataFrame(rows)

winner_participation = winner_participation_table(validated_weights, WINNER_TICKERS)
winner_participation.to_parquet(output_dir / "winner_participation.parquet", index=False)
print("Winner participation:")
display(winner_participation)

score_validation_table.to_parquet(output_dir / "router_score_validation_table.parquet", index=False)
calibration_table.to_parquet(output_dir / "router_calibration_table.parquet", index=False)

extra_artifact_paths = {
    "router_candidate_comparison": str(output_dir / "router_candidate_comparison.parquet"),
    "router_stress_table": str(output_dir / "router_stress_table.parquet"),
    "router_regime_return_table": str(output_dir / "router_regime_return_table.parquet"),
    "router_risk_log": str(output_dir / "router_risk_log.parquet"),
    "winner_participation": str(output_dir / "winner_participation.parquet"),
    "router_score_validation_table": str(output_dir / "router_score_validation_table.parquet"),
    "router_calibration_table": str(output_dir / "router_calibration_table.parquet"),
}
artifact_paths.update(extra_artifact_paths)
result.artifact_paths.update(extra_artifact_paths)

print("Selected candidate:", selected_candidate_name)
display(backtest_comparison.head(12))
print("Final stress table:")
display(final_stress_table)
print("Regime return table:")
display(final_regime_return_table)

Running backtest: weekly_multi_regime_router_score_hvw005_beta085
Winner participation:


,ticker,available,held_share,avg_weight_when_held,max_weight
0,PLTR,False,0.0,0.000000,0.000000
1,HOOD,False,0.0,0.000000,0.000000
2,NVDA,True,1.0,0.022429,0.049239
3,AVGO,True,1.0,0.029114,0.049691
4,KTOS,False,0.0,0.000000,0.000000
5,BWXT,False,0.0,0.000000,0.000000


Selected candidate: weekly_multi_regime_router_score_hvw005_beta085


,candidate,annual_return,annual_volatility,sharpe,sortino,max_drawdown,average_turnover,total_return,stress_worst_day,stress_worst_5_day,stress_crash_day_avg,beta_to_benchmark,avg_active_names,avg_effective_names,avg_max_weight,avg_beta_estimate,state_counts,router_selection_score
0,weekly_multi_regime_router_score_hvw005_beta085,0.174853,0.278686,0.627419,0.923936,-0.379052,0.020882,0.899298,-0.066467,-0.132797,-0.029564,1.431002,25.0,22.874636,0.047308,1.42319,"{""rebound"": 6, ""stress"": 202}",0.699394


Final stress table:


,stress_metric,value
0,realized_volatility,0.279912
1,beta_to_benchmark,1.431002
2,worst_day,-0.066467
3,worst_5_day_period,-0.132797
4,worst_month,-0.144708
5,avg_return_benchmark_worst_decile_days,-0.029564
6,avg_return_benchmark_rebound_days,0.028785
7,benchmark_worst_decile_threshold,-0.012386


Regime return table:


,bucket,observations,avg_strategy_return,strategy_vol_ann,avg_benchmark_return,hit_rate
0,benchmark_up_days,538,0.011666,0.205891,0.007847,0.877323
1,benchmark_down_days,454,-0.012052,0.210674,-0.008179,0.167401
2,high_benchmark_vol,536,0.000924,0.337028,0.000510,0.539179
3,benchmark_worst_decile_days,100,-0.029564,0.191316,-0.020410,0.000000
4,benchmark_rebound_days,100,0.028785,0.260751,0.019959,1.000000
5,market_drawdown_regime,399,-0.002691,0.343199,-0.001698,0.458647
6,market_high_vol_regime,540,0.002811,0.200466,0.001759,0.618519
7,benchmark_agnostic_stress,572,-0.001560,0.329404,-0.000842,0.472028


## Reusable Inference, Checkpoint, And Artifact Reload Test

In [14]:
def make_inference_sequences(feature_frame: pd.DataFrame, X: np.ndarray, seq_len: int) -> tuple[np.ndarray, pd.DataFrame]:
    Xs: list[np.ndarray] = []
    meta: list[dict[str, object]] = []
    working = feature_frame.sort_values(["ticker", "date"]).reset_index(drop=True)
    for ticker, group in working.groupby("ticker", sort=False):
        positions = group.index.to_numpy(dtype=int)
        if len(positions) < seq_len:
            continue
        dates = pd.to_datetime(group["date"]).to_numpy()
        for end_offset in range(seq_len - 1, len(positions)):
            window_positions = positions[end_offset - seq_len + 1 : end_offset + 1]
            row = group.iloc[end_offset]
            Xs.append(X[window_positions])
            meta.append(
                {
                    "date": pd.Timestamp(dates[end_offset]),
                    "ticker": ticker,
                    "historical_vol_20d": float(np.clip(row["vol_20d"] * np.sqrt(252.0), 1e-4, None)),
                    "spy_vol_20d_ann": float(row["spy_vol_20d_ann"]),
                    "spy_drawdown_60d": float(row["spy_drawdown_60d"]),
                    "spy_momentum_20d": float(row["spy_momentum_20d"]),
                    "beta_60d_spy": float(row["beta_60d_spy"]),
                    "benchmark_agnostic_stress": float(row["benchmark_agnostic_stress"]),
                    "avg_pairwise_corr_20d": float(row["avg_pairwise_corr_20d"]),
                    "bench_vol_20d_ann": float(row["bench_vol_20d_ann"]),
                    "bench_drawdown_60d": float(row["bench_drawdown_60d"]),
                    "bench_momentum_20d": float(row["bench_momentum_20d"]),
                    "bench_rebound_state": float(row["bench_rebound_state"]),
                    "bench_stress_state": float(row["bench_stress_state"]),
                    "beta_60d_benchmark": float(row["beta_60d_benchmark"]),
                    "dual_context_vol_spread": float(row["dual_context_vol_spread"]),
                    "rebound_breadth_confirmation": float(row["rebound_breadth_confirmation"]),
                }
            )
    if not Xs:
        return np.empty((0, seq_len, len(ALL_FEATURE_NAMES)), dtype=np.float32), pd.DataFrame(meta)
    return np.stack(Xs).astype(np.float32), pd.DataFrame(meta)


def _series_from_mapping(value) -> pd.Series:
    return value if isinstance(value, pd.Series) else pd.Series(value, dtype=float)


def predict_from_prices(model_bundle: dict, prices: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    model_obj = model_bundle["model"]
    model_device = model_bundle.get("device", device)
    feature_names = list(model_bundle["feature_names"])
    seq_len = int(model_bundle["seq_len"])
    horizon = int(model_bundle["horizon"])
    train_mean = _series_from_mapping(model_bundle["train_means"])
    train_std = _series_from_mapping(model_bundle["train_stds"])
    t_means = dict(model_bundle["target_means"])
    t_stds = dict(model_bundle["target_stds"])

    features = build_model_features(prices).replace([np.inf, -np.inf], np.nan)
    if tickers is not None:
        tickers_normalized = [str(ticker).upper() for ticker in tickers]
        features = features.loc[features["ticker"].isin(tickers_normalized)].copy()
    features = features.dropna(subset=feature_names).sort_values(["ticker", "date"]).reset_index(drop=True)
    if features.empty:
        return pd.DataFrame(columns=["date", "ticker", "horizon", "expected_return"])

    X = ((features[feature_names] - train_mean.loc[feature_names]) / train_std.loc[feature_names]).to_numpy(dtype=np.float32)
    X_infer, meta = make_inference_sequences(features, X, seq_len)
    if len(meta) == 0:
        return pd.DataFrame(columns=["date", "ticker", "horizon", "expected_return"])

    model_obj.eval()
    returns, alphas, vols, tails, downside_probs, regime_probs, latent_norms = [], [], [], [], [], [], []
    with torch.no_grad():
        for start in range(0, len(X_infer), 4096):
            xb = torch.as_tensor(X_infer[start : start + 4096], dtype=torch.float32, device=model_device)
            output = model_obj(xb)
            returns.append(output["expected_return"].detach().cpu().numpy())
            alphas.append(output["expected_alpha"].detach().cpu().numpy())
            vols.append(output["expected_vol_log"].detach().cpu().numpy())
            tails.append(output["expected_tail_log"].detach().cpu().numpy())
            downside_probs.append(torch.sigmoid(output["downside_logit"]).detach().cpu().numpy())
            regime_probs.append(torch.softmax(output["regime_logits"], dim=1).detach().cpu().numpy())
            latent_norms.append(torch.linalg.norm(output["latent"], dim=1).detach().cpu().numpy())

    scored = meta.copy()
    scored["expected_return"] = np.concatenate(returns) * t_stds["return"] + t_means["return"]
    scored["expected_alpha"] = np.concatenate(alphas) * t_stds["alpha"] + t_means["alpha"]
    scored["expected_forward_volatility"] = np.expm1(np.concatenate(vols) * t_stds["vol_log"] + t_means["vol_log"]).clip(min=1e-4)
    scored["expected_tail_loss"] = np.expm1(np.concatenate(tails) * t_stds["tail_log"] + t_means["tail_log"]).clip(min=0.0)
    scored["expected_downside_risk"] = np.concatenate(downside_probs)
    scored["expected_volatility"] = np.clip(scored["historical_vol_20d"].to_numpy(dtype=float), 1e-4, None)
    scored["uncertainty"] = np.abs(scored["expected_return"] - scored["expected_alpha"])
    regime_array = np.concatenate(regime_probs)
    for idx, name in enumerate(model_bundle["regime_classes"]):
        scored[f"regime_prob_{name}"] = regime_array[:, idx]
    scored["regime_confidence"] = scored["regime_prob_calm_risk_on"] + 0.50 * scored["regime_prob_rebound"] - 0.75 * scored["regime_prob_high_vol"] - scored["regime_prob_drawdown"]
    scored = add_scores(scored)
    scored["horizon"] = horizon

    if dates is not None:
        requested_dates = pd.to_datetime(pd.Series(dates), utc=True).dt.tz_localize(None)
        scored = scored.loc[scored["date"].isin(set(requested_dates))].copy()

    keep_cols = [
        "date", "ticker", "horizon", "expected_return", "expected_alpha", "expected_volatility", "expected_forward_volatility",
        "expected_tail_loss", "expected_downside_risk", "uncertainty", "regime_confidence",
        "regime_prob_calm_risk_on", "regime_prob_high_vol", "regime_prob_drawdown", "regime_prob_rebound",
        "spy_vol_20d_ann", "spy_drawdown_60d", "spy_momentum_20d", "beta_60d_spy", "benchmark_agnostic_stress",
        "bench_vol_20d_ann", "bench_drawdown_60d", "bench_momentum_20d", "bench_rebound_state", "bench_stress_state", "beta_60d_benchmark", "dual_context_vol_spread", "rebound_breadth_confirmation",
        "benchmark_beta_excess",
        "alpha_hist_vol_score", "alpha_pred_vol_score", "vol_blended_score", "strong_downside_score", "rebound_capture_score", "state_machine_score", "multi_regime_router_score",
        "alpha_hist_vol_score_rank", "alpha_pred_vol_score_rank", "vol_blended_score_rank", "strong_downside_score_rank", "rebound_capture_score_rank", "state_machine_score_rank", "multi_regime_router_score_rank",
    ]
    return scored.loc[:, keep_cols].reset_index(drop=True)


model_config = {
    "architecture": "MultiRegimeRouterLSTMAutoencoder",
    "input_dim": len(ALL_FEATURE_NAMES),
    "hidden_size": HIDDEN_SIZE,
    "latent_dim": LATENT_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "regime_classes": len(REGIME_CLASSES),
    "seq_len": SEQ_LEN,
}
checkpoint = {
    "state_dict": model.state_dict(),
    "model_config": model_config,
    "feature_names": ALL_FEATURE_NAMES,
    "base_feature_names": BASE_FEATURE_NAMES,
    "custom_feature_names": CUSTOM_FEATURE_NAMES,
    "train_means": train_means.to_dict(),
    "train_stds": train_stds.to_dict(),
    "target_means": target_means,
    "target_stds": target_stds,
    "regime_classes": REGIME_CLASSES,
    "regime_thresholds": regime_thresholds,
    "reconstruction_feature_weights": reconstruction_feature_weights.to_dict(),
    "research_end": str(RESEARCH_END.date()),
    "benchmark_context_ticker": BENCHMARK_CONTEXT_TICKER,
    "selected_candidate": selected_candidate_name,
    "winner_tickers": WINNER_TICKERS,
    "scoring_config": {
        "primary_score": "multi_regime_router_score",
        "score_variants": score_columns,
        "multi_regime_router_score": True,
        "predicted_forward_vol_is_denominator": True,
        "downside_score_penalty": DOWNSIDE_SCORE_PENALTY,
        "tail_score_penalty": TAIL_SCORE_PENALTY,
        "uncertainty_score_penalty": UNCERTAINTY_SCORE_PENALTY,
    },
    "portfolio_config": {
        "portfolio_builder": "weights_from_multiregime_router_scores",
        "selected_candidate": selected_candidate_name,
        "beta_target_grid": BETA_TARGET_GRID,
    "winner_tickers": WINNER_TICKERS,
        "benchmark_context_ticker": BENCHMARK_CONTEXT_TICKER,
        "normal_max_weight": NORMAL_MAX_WEIGHT,
        "high_vol_max_weight_options": HIGH_VOL_MAX_WEIGHT_OPTIONS,
        "high_risk_min_active_names": HIGH_RISK_MIN_ACTIVE_NAMES,
        "normal_min_active_names": NORMAL_MIN_ACTIVE_NAMES,
        "turnover_blend": TURNOVER_BLEND,
        "beta_target": BETA_TARGET,
        "minvar_anchor": True,
        "cov_lookback_days": COV_LOOKBACK_DAYS,
    },
    "horizon": HORIZON,
    "rebalance_frequency": REBALANCE_FREQUENCY,
    "random_seed": RANDOM_SEED,
}
model_artifact_path = output_dir / "supervised_lstm_autoencoder_multiregime_router.pt"
torch.save(checkpoint, model_artifact_path)
print("Saved model checkpoint:", model_artifact_path)

model_bundle = {
    "model": model,
    "device": device,
    "feature_names": ALL_FEATURE_NAMES,
    "train_means": train_means,
    "train_stds": train_stds,
    "target_means": target_means,
    "target_stds": target_stds,
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
    "regime_classes": REGIME_CLASSES,
}

weekly_dates = pd.DatetimeIndex(validated_weights.index)
submission_style_predictions = predict_from_prices(model_bundle, prices, dates=weekly_dates, tickers=validated_weights.columns)
submission_style_predictions = validate_prediction_frame(submission_style_predictions, dataset_name=research_spec, horizon=HORIZON, repo_root=repo_root)


def load_model_bundle_from_checkpoint(checkpoint_path: Path, map_location="cpu") -> dict[str, object]:
    payload = torch.load(checkpoint_path, map_location=map_location)
    cfg = dict(payload["model_config"])
    model_obj = MultiRegimeRouterLSTMAutoencoder(
        input_dim=cfg["input_dim"],
        hidden_size=cfg["hidden_size"],
        latent_dim=cfg["latent_dim"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        regime_classes=cfg["regime_classes"],
    )
    model_obj.load_state_dict(payload["state_dict"])
    model_obj.eval()
    return {
        "model": model_obj,
        "device": torch.device(map_location),
        "feature_names": payload["feature_names"],
        "train_means": pd.Series(payload["train_means"], dtype=float),
        "train_stds": pd.Series(payload["train_stds"], dtype=float),
        "target_means": payload["target_means"],
        "target_stds": payload["target_stds"],
        "seq_len": payload["model_config"]["seq_len"],
        "horizon": payload["horizon"],
        "regime_classes": payload["regime_classes"],
    }


reloaded_bundle = load_model_bundle_from_checkpoint(model_artifact_path, map_location="cpu")
reload_predictions = predict_from_prices(reloaded_bundle, prices, dates=weekly_dates[: min(3, len(weekly_dates))], tickers=validated_weights.columns)
reload_predictions = validate_prediction_frame(reload_predictions, dataset_name=research_spec, horizon=HORIZON, repo_root=repo_root)
print("Reusable inference predictions:", submission_style_predictions.shape)
print("Artifact reload predictions:", reload_predictions.shape)
display(submission_style_predictions.head())

Saved model checkpoint: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/runs/supervised_lstm_autoencoder_multiregime_router_shared_set_2/supervised_lstm_autoencoder_multiregime_router.pt
Reusable inference predictions: (5198, 43)
Artifact reload predictions: (75, 43)


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,expected_forward_volatility,expected_tail_loss,expected_downside_risk,uncertainty,...,rebound_capture_score,state_machine_score,multi_regime_router_score,alpha_hist_vol_score_rank,alpha_pred_vol_score_rank,vol_blended_score_rank,strong_downside_score_rank,rebound_capture_score_rank,state_machine_score_rank,multi_regime_router_score_rank
0,2022-01-07,AAPL,5,0.008404,0.006818,0.291344,0.233316,0.017686,0.100993,0.001586,...,-0.110607,-0.562178,-0.621337,0.04,0.52,0.92,0.52,0.92,0.52,0.60
1,2022-01-07,ADBE,5,0.005455,0.003928,0.519157,0.254422,0.020204,0.124629,0.001527,...,-0.132217,-0.680802,-0.922686,-0.92,-0.92,-0.36,-0.36,-0.52,-0.36,-0.60
2,2022-01-07,ADI,5,0.007133,0.005582,0.242832,0.253624,0.019539,0.121196,0.001551,...,-0.128256,-0.587835,-0.673255,-0.04,-0.44,0.20,0.20,-0.28,0.20,0.28
3,2022-01-07,AMAT,5,0.009963,0.009855,0.408457,0.277097,0.021766,0.127297,0.000108,...,-0.120832,-0.703052,-0.897530,0.20,0.68,0.12,-0.60,0.28,-0.60,-0.44
4,2022-01-07,AMD,5,0.014073,0.016175,0.588302,0.320840,0.024107,0.166784,0.002102,...,-0.125625,-0.799054,-1.102699,0.60,0.92,-0.68,-0.76,0.04,-0.76,-0.76


## MLflow Logging

In [ ]:
def mlflow_tracking_preflight(tracking_uri: str, timeout: float = 10.0) -> tuple[bool, str]:
    from urllib.error import HTTPError, URLError
    from urllib.parse import urlparse
    from urllib.request import Request, urlopen
    import socket

    parsed = urlparse(tracking_uri)
    if parsed.scheme not in {"http", "https"}:
        return True, "local tracking URI"
    if not parsed.hostname:
        return False, f"remote tracking URI has no hostname: {tracking_uri}"
    proxy_env = ["HTTPS_PROXY", "HTTP_PROXY", "ALL_PROXY", "https_proxy", "http_proxy", "all_proxy"]
    configured_proxy = next((name for name in proxy_env if os.environ.get(name)), None)
    if configured_proxy:
        try:
            request = Request(tracking_uri.rstrip("/") + "/", method="GET")
            with urlopen(request, timeout=timeout):
                return True, f"{parsed.hostname} reachable through {configured_proxy}"
        except HTTPError as exc:
            return True, f"{parsed.hostname} reachable through {configured_proxy}; HTTP {exc.code}"
        except URLError as exc:
            return False, f"{parsed.hostname} not reachable through {configured_proxy} ({exc.reason})"
        except OSError as exc:
            return False, f"{parsed.hostname} not reachable through {configured_proxy} ({exc})"
    port = parsed.port or (443 if parsed.scheme == "https" else 80)
    try:
        with socket.create_connection((parsed.hostname, port), timeout=timeout):
            return True, f"{parsed.hostname}:{port} reachable"
    except OSError as exc:
        return False, f"{parsed.hostname}:{port} not reachable ({exc})"


if LOG_TO_MLFLOW:
    mlflow_layout = init_mlflow(repo_root)
    tracking_uri = mlflow_layout["tracking_uri"]
    print("MLflow tracking URI:", tracking_uri)
    tracking_ok, tracking_reason = mlflow_tracking_preflight(tracking_uri)
    if not tracking_ok:
        message = f"MLflow tracking preflight failed: {tracking_reason}"
        if MLFLOW_REQUIRED:
            raise ConnectionError(message)
        print("MLflow logging skipped:", message)
    else:
        try:
            with start_run(
                run_name=RUN_NAME,
                dataset_name=DATASET_NAME,
                tags={
                    "workflow": "multi_regime_volatility_router",
                    "model_family": "torch_lstm_autoencoder",
                    "prediction_horizon": str(HORIZON),
                    "rebalance_frequency": REBALANCE_FREQUENCY,
                    "research_end": str(RESEARCH_END.date()),
                },
                repo_root=repo_root,
            ):
                import mlflow
                mlflow.log_params(
                    {
                        "model_name": MODEL_NAME,
                        "dataset_name": DATASET_NAME,
                        "research_end": str(RESEARCH_END.date()),
                        "horizon": HORIZON,
                        "seq_len": SEQ_LEN,
                        "feature_count": len(ALL_FEATURE_NAMES),
                        "latent_dim": LATENT_DIM,
                        "hidden_size": HIDDEN_SIZE,
                        "num_layers": NUM_LAYERS,
                        "dropout": DROPOUT,
                        "epochs_requested": EPOCHS,
                        "epochs_run": len(history),
                        "selected_candidate": selected_candidate_name,
                        "benchmark_context_ticker": BENCHMARK_CONTEXT_TICKER,
                        "portfolio_score": portfolio.metadata["score_col"],
                        "portfolio_high_vol_max_weight": portfolio.metadata["high_vol_max_weight"],
                        "portfolio_beta_target": portfolio.metadata.get("beta_target", BETA_TARGET),
                        "avg_active_names": float((validated_weights > 1e-8).sum(axis=1).mean()),
                        "avg_effective_names": float(risk_log["effective_names"].mean()),
                        "avg_realized_max_weight": float(risk_log["realized_max_weight"].mean()),
                        "avg_beta_estimate": float(risk_log["portfolio_beta_estimate"].mean()),
                        "avg_beta_target_gap": float(risk_log["beta_target_gap"].mean()),
                        "low_confidence_fallback_share": float(risk_log["low_confidence_fallback"].mean()),
                        "val_return_mse": val_return_mse,
                        "val_alpha_mse": val_alpha_mse,
                        "val_vol_mse": val_vol_mse,
                        "val_tail_mse": val_tail_mse,
                        "val_downside_brier": val_downside_brier,
                        "val_regime_accuracy": val_regime_accuracy,
                    }
                )
                for _, row in final_stress_table.iterrows():
                    mlflow.log_metric(f"stress_{row['stress_metric']}", float(row["value"]))
                log_predictions(predictions)
                log_portfolio(portfolio)
                log_backtest(result)
                manifest = log_model_submission(
                    {"model_checkpoint": model_artifact_path},
                    model_name=MODEL_NAME,
                    model_family="torch",
                    feature_names=ALL_FEATURE_NAMES,
                    target=alpha_target_col,
                    horizon=HORIZON,
                    rebalance_frequency=REBALANCE_FREQUENCY,
                    preprocessing={
                        "scaler": "train_window_mean_std_non_benchmark_rows",
                        "target_scaler": "return_alpha_forward_vol_tail_mean_std_non_benchmark_rows",
                        "train_means": train_means.to_dict(),
                        "train_stds": train_stds.to_dict(),
                        "target_means": target_means,
                        "target_stds": target_stds,
                        "regime_thresholds": regime_thresholds,
                        "regime_classes": REGIME_CLASSES,
                    },
                    model_config={
                        **model_config,
                        "portfolio_builder": "weights_from_multiregime_router_scores",
                        "benchmark_context_ticker": BENCHMARK_CONTEXT_TICKER,
                        "required_functions": ["build_model_features", "predict_from_prices"],
                        "portfolio": portfolio.metadata,
                        "scoring_config": checkpoint["scoring_config"],
                        "artifact_reload_test": "passed",
                    },
                    source_files=[NOTEBOOK_PATH] if NOTEBOOK_PATH.exists() else None,
                    notes="Multi-regime volatility router LSTM-AE with dataset-benchmark/SPY context, benchmark-beta controls, state-machine scoring, hard tail-risk filtering, rebound gate, beta target sweeps, calibration tables, and min-variance anchoring.",
                )
                print("MLflow logging complete. Submission manifest keys:", sorted(manifest.keys()))
        except Exception as exc:
            if MLFLOW_REQUIRED:
                raise
            print("MLflow logging skipped after error:", repr(exc))
else:
    message = "MLflow logging disabled because SKIP_MLFLOW=1."
    if MLFLOW_REQUIRED:
        raise RuntimeError(message)
    print(message)

## Final Checks

In [ ]:
assert {"total_return", "annual_return", "annual_volatility", "sharpe", "max_drawdown"}.issubset(result.metrics)
assert pd.Timestamp(prices["date"].max()) <= RESEARCH_END, "Local research data should not include dates after 2022."
assert set(validated_weights.columns).isdisjoint({research_spec.benchmark_ticker}), "Benchmark ticker should not be traded."
assert validated_weights.index.max() <= RESEARCH_END, "Local backtest weights should not extend beyond the research cap."
assert validated_weights.index.is_monotonic_increasing
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert predictions["expected_forward_volatility"].notna().all()
assert predictions["expected_tail_loss"].notna().all()
assert predictions["expected_downside_risk"].between(0.0, 1.0).all()
assert predictions["ticker"].ne(research_spec.benchmark_ticker).all()
assert Path(artifact_paths["quantstats_report"]).exists()
assert model_artifact_path.exists()
loaded_payload = torch.load(model_artifact_path, map_location="cpu")
assert "state_dict" in loaded_payload
assert loaded_payload["feature_names"] == ALL_FEATURE_NAMES
assert len(submission_style_predictions) == len(predictions)
assert len(reload_predictions) > 0

assert "market_state" in risk_log.columns
assert Path(output_dir / "winner_participation.parquet").exists()
assert Path(output_dir / "router_calibration_table.parquet").exists()
print("End-to-end multi-regime volatility router workflow validated successfully.")
print("Selected candidate:", selected_candidate_name)
print("Key risk metrics:")
for key in ["annual_return", "annual_volatility", "sharpe", "sortino", "max_drawdown", "average_turnover"]:
    print(f"  {key:<20s}: {result.metrics[key]:.6f}")
print("Average active names:", float((validated_weights > 1e-8).sum(axis=1).mean()))
print("Average effective names:", float(risk_log["effective_names"].mean()))
print("Average max weight:", float(risk_log["realized_max_weight"].mean()))
print("Average beta estimate:", float(risk_log["portfolio_beta_estimate"].mean()))
print("Average beta target gap:", float(risk_log["beta_target_gap"].mean()))

display(result.nav.tail().to_frame("nav"))
display(risk_log.tail())

## Four-Regime Proxy Dataset Backtests

This section runs Hannah's already-trained multi-regime router against the four proxy regime datasets in `configs/datasets.toml`. Each backtest is allocated `$1.2M`; the final table reports per-regime ending value plus the combined `$4.8M` bankroll return.


In [15]:
from datetime import timedelta
from IPython.display import display
import yfinance as yf

REGIME_BACKTEST_DATASETS = [
    "regime_modern_tech_gain_2022_2026",
    "regime_financial_crisis_loss_2005_2010",
    "regime_nineties_volatility_1995_1999",
    "regime_oil_pre2014_energy_2010_2013",
]
REGIME_BACKTEST_ALLOCATION = 1_200_000.0
REGIME_BACKTEST_TOTAL_BANKROLL = REGIME_BACKTEST_ALLOCATION * len(REGIME_BACKTEST_DATASETS)
REGIME_BACKTEST_OUTPUT_DIR = output_dir / "four_regime_proxy_backtests"
REGIME_BACKTEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _normalize_extra_yfinance_frame(frame: pd.DataFrame, ticker: str) -> pd.DataFrame:
    normalized = frame.copy()
    if isinstance(normalized.columns, pd.MultiIndex):
        normalized.columns = normalized.columns.get_level_values(0)
    normalized.columns.name = None
    if "Adj Close" not in normalized.columns and "Close" in normalized.columns:
        normalized["Adj Close"] = normalized["Close"]
    normalized = normalized.reset_index().rename(
        columns={
            "Date": "date",
            "Datetime": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume",
        }
    )
    normalized["ticker"] = ticker.upper()
    normalized = normalized.loc[:, ["date", "ticker", "open", "high", "low", "close", "adj_close", "volume"]]
    normalized["date"] = pd.to_datetime(normalized["date"], utc=True).dt.tz_localize(None)
    for col in ["open", "high", "low", "close", "adj_close", "volume"]:
        normalized[col] = pd.to_numeric(normalized[col], errors="coerce")
    return normalized.dropna(subset=["date", "adj_close"]).sort_values(["ticker", "date"]).reset_index(drop=True)


def _download_extra_context_ticker(ticker: str, spec) -> pd.DataFrame:
    start = pd.Timestamp(spec.start_date).date().isoformat()
    end = (pd.Timestamp(spec.end_date).date() + timedelta(days=1)).isoformat()
    downloaded = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        threads=False,
    )
    if downloaded.empty:
        raise ValueError(f"No yfinance rows downloaded for required context ticker {ticker} in {spec.identifier}.")
    return _normalize_extra_yfinance_frame(downloaded, ticker)


def _prices_with_spy_context(prices_frame: pd.DataFrame, spec) -> pd.DataFrame:
    prices_for_features = prices_frame.copy()
    prices_for_features["date"] = pd.to_datetime(prices_for_features["date"], utc=True).dt.tz_localize(None)
    prices_for_features["ticker"] = prices_for_features["ticker"].astype(str).str.upper()
    if "SPY" not in set(prices_for_features["ticker"]):
        spy_context = _download_extra_context_ticker("SPY", spec)
        prices_for_features = pd.concat([prices_for_features, spy_context], ignore_index=True)
    return prices_for_features.sort_values(["ticker", "date"]).reset_index(drop=True)


def _selected_portfolio_kwargs() -> dict[str, object]:
    selected_meta = dict(getattr(portfolio, "metadata", {}) or {})
    return {
        "score_col": selected_meta.get("score_col", "multi_regime_router_score_rank"),
        "frequency": selected_meta.get("rebalance_frequency", REBALANCE_FREQUENCY),
        "high_vol_max_weight": float(selected_meta.get("high_vol_max_weight", min(HIGH_VOL_MAX_WEIGHT_OPTIONS))),
        "normal_max_weight": float(selected_meta.get("normal_max_weight", NORMAL_MAX_WEIGHT)),
        "turnover_blend": float(selected_meta.get("turnover_blend", TURNOVER_BLEND)),
        "beta_target": float(selected_meta.get("beta_target", BETA_TARGET)),
    }


def _regime_rebalance_dates(prices_frame: pd.DataFrame, spec, frequency: str) -> pd.DatetimeIndex:
    dates = pd.to_datetime(prices_frame["date"], utc=True).dt.tz_localize(None)
    mask = (dates >= pd.Timestamp(spec.test_start)) & (dates <= pd.Timestamp(spec.test_end))
    test_dates = pd.DatetimeIndex(dates.loc[mask].drop_duplicates().sort_values())
    if test_dates.empty:
        raise ValueError(f"No trading dates found for {spec.identifier} test window: {spec.test_start} to {spec.test_end}")
    return select_rebalance_dates(test_dates, frequency=frequency)


def _run_hannah_regime_backtest(dataset_name: str) -> dict[str, object]:
    global BENCHMARK_CONTEXT_TICKER, price_wide, returns_wide

    spec = get_dataset_spec(dataset_name, repo_root=repo_root)
    tradable = [ticker for ticker in spec.tickers if ticker != spec.benchmark_ticker]
    if not tradable:
        raise ValueError(f"{dataset_name} has no tradable tickers after excluding the benchmark.")

    loaded_prices = load_prices(spec, repo_root=repo_root)
    loaded_prices["date"] = pd.to_datetime(loaded_prices["date"], utc=True).dt.tz_localize(None)
    loaded_prices["ticker"] = loaded_prices["ticker"].astype(str).str.upper()
    feature_prices = _prices_with_spy_context(loaded_prices, spec)

    kwargs = _selected_portfolio_kwargs()
    rebalance_dates = _regime_rebalance_dates(loaded_prices, spec, kwargs["frequency"])

    old_context_ticker = BENCHMARK_CONTEXT_TICKER
    old_price_wide = price_wide
    old_returns_wide = returns_wide
    try:
        BENCHMARK_CONTEXT_TICKER = spec.benchmark_ticker.upper()
        price_wide = _price_wide(loaded_prices, tradable)
        returns_wide = price_wide.pct_change()

        regime_predictions = predict_from_prices(
            model_bundle,
            feature_prices,
            dates=rebalance_dates,
            tickers=tradable,
        )
        regime_predictions = regime_predictions.loc[
            (pd.to_datetime(regime_predictions["date"]) >= pd.Timestamp(spec.test_start))
            & (pd.to_datetime(regime_predictions["date"]) <= pd.Timestamp(spec.test_end))
        ].reset_index(drop=True)
        if regime_predictions.empty:
            raise ValueError(f"No predictions produced for {dataset_name}. Check feature coverage and context ticker data.")
        regime_predictions = validate_prediction_frame(
            regime_predictions,
            dataset_name=spec,
            horizon=HORIZON,
            repo_root=repo_root,
        )

        regime_portfolio, regime_risk_log = weights_from_multiregime_router_scores(
            regime_predictions,
            dataset_name=spec,
            strategy_name=f"{MODEL_NAME}_{dataset_name}",
            universe_tickers=tradable,
            **kwargs,
        )
        regime_portfolio.weights = validate_weights_frame(regime_portfolio.weights, dataset_name=spec, repo_root=repo_root)
        regime_result = backtest_weights(spec, regime_portfolio, benchmark=spec.benchmark_ticker, repo_root=repo_root)
        regime_result.metrics = build_metrics(regime_result)
        regime_stress = stress_table(regime_result, spec.benchmark_ticker)
    finally:
        BENCHMARK_CONTEXT_TICKER = old_context_ticker
        price_wide = old_price_wide
        returns_wide = old_returns_wide

    dataset_output_dir = REGIME_BACKTEST_OUTPUT_DIR / dataset_name
    dataset_output_dir.mkdir(parents=True, exist_ok=True)
    regime_predictions.to_parquet(dataset_output_dir / "predictions.parquet", index=False)
    regime_portfolio.weights.to_parquet(dataset_output_dir / "weights.parquet")
    regime_risk_log.to_parquet(dataset_output_dir / "risk_log.parquet", index=False)
    regime_stress.to_parquet(dataset_output_dir / "stress_table.parquet", index=False)
    regime_result.artifact_paths.update(write_artifacts_for_notebook(regime_result, dataset_output_dir))

    return {
        "dataset_name": dataset_name,
        "spec": spec,
        "predictions": regime_predictions,
        "portfolio": regime_portfolio,
        "risk_log": regime_risk_log,
        "stress_table": regime_stress,
        "result": regime_result,
        "output_dir": dataset_output_dir,
    }


In [16]:
regime_backtest_runs = {}
for dataset_name in REGIME_BACKTEST_DATASETS:
    print(f"Running Hannah four-regime backtest: {dataset_name}", flush=True)
    regime_backtest_runs[dataset_name] = _run_hannah_regime_backtest(dataset_name)

print("Completed regime backtests:", list(regime_backtest_runs))


Running Hannah four-regime backtest: regime_modern_tech_gain_2022_2026
Running Hannah four-regime backtest: regime_financial_crisis_loss_2005_2010
Running Hannah four-regime backtest: regime_nineties_volatility_1995_1999
Running Hannah four-regime backtest: regime_oil_pre2014_energy_2010_2013
Completed regime backtests: ['regime_modern_tech_gain_2022_2026', 'regime_financial_crisis_loss_2005_2010', 'regime_nineties_volatility_1995_1999', 'regime_oil_pre2014_energy_2010_2013']


In [17]:
summary_rows = []
for dataset_name, payload in regime_backtest_runs.items():
    spec = payload["spec"]
    result_obj = payload["result"]
    risk_log_obj = payload["risk_log"]
    stress_obj = payload["stress_table"]
    metrics_obj = dict(result_obj.metrics)
    total_return = float(metrics_obj.get("total_return", np.nan))
    ending_bankroll = REGIME_BACKTEST_ALLOCATION * (1.0 + total_return)
    stress_map = dict(zip(stress_obj["stress_metric"], stress_obj["value"])) if not stress_obj.empty else {}
    summary_rows.append(
        {
            "dataset_name": dataset_name,
            "benchmark_ticker": spec.benchmark_ticker,
            "test_start": spec.test_start,
            "test_end": spec.test_end,
            "allocation": REGIME_BACKTEST_ALLOCATION,
            "ending_bankroll": ending_bankroll,
            "profit_loss": ending_bankroll - REGIME_BACKTEST_ALLOCATION,
            "total_return": total_return,
            "annual_return": metrics_obj.get("annual_return", np.nan),
            "annual_volatility": metrics_obj.get("annual_volatility", np.nan),
            "sharpe": metrics_obj.get("sharpe", np.nan),
            "sortino": metrics_obj.get("sortino", np.nan),
            "max_drawdown": metrics_obj.get("max_drawdown", np.nan),
            "average_turnover": metrics_obj.get("average_turnover", np.nan),
            "avg_active_names": float((result_obj.weights > 1e-8).sum(axis=1).mean()),
            "avg_effective_names": float(risk_log_obj["effective_names"].mean()) if not risk_log_obj.empty else np.nan,
            "avg_realized_max_weight": float(risk_log_obj["realized_max_weight"].mean()) if not risk_log_obj.empty else np.nan,
            "avg_beta_estimate": float(risk_log_obj["portfolio_beta_estimate"].mean()) if not risk_log_obj.empty else np.nan,
            "stress_worst_day": stress_map.get("worst_day", np.nan),
            "stress_worst_5_day": stress_map.get("worst_5_day_period", np.nan),
            "stress_beta_to_benchmark": stress_map.get("beta_to_benchmark", np.nan),
        }
    )

regime_backtest_summary = pd.DataFrame(summary_rows)
combined_ending_bankroll = float(regime_backtest_summary["ending_bankroll"].sum())
combined_profit_loss = combined_ending_bankroll - REGIME_BACKTEST_TOTAL_BANKROLL
combined_total_return = combined_profit_loss / REGIME_BACKTEST_TOTAL_BANKROLL
combined_row = pd.DataFrame(
    [
        {
            "dataset_name": "COMBINED_4_REGIME_BANKROLL",
            "benchmark_ticker": "mixed",
            "test_start": min(regime_backtest_summary["test_start"]),
            "test_end": max(regime_backtest_summary["test_end"]),
            "allocation": REGIME_BACKTEST_TOTAL_BANKROLL,
            "ending_bankroll": combined_ending_bankroll,
            "profit_loss": combined_profit_loss,
            "total_return": combined_total_return,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan,
            "average_turnover": np.nan,
            "avg_active_names": np.nan,
            "avg_effective_names": np.nan,
            "avg_realized_max_weight": np.nan,
            "avg_beta_estimate": np.nan,
            "stress_worst_day": np.nan,
            "stress_worst_5_day": np.nan,
            "stress_beta_to_benchmark": np.nan,
        }
    ]
)
regime_backtest_summary_with_total = pd.concat([regime_backtest_summary, combined_row], ignore_index=True)

summary_path = REGIME_BACKTEST_OUTPUT_DIR / "four_regime_bankroll_summary.parquet"
csv_summary_path = REGIME_BACKTEST_OUTPUT_DIR / "four_regime_bankroll_summary.csv"
regime_backtest_summary_with_total.to_parquet(summary_path, index=False)
regime_backtest_summary_with_total.to_csv(csv_summary_path, index=False)

print(f"Starting bankroll: ${REGIME_BACKTEST_TOTAL_BANKROLL:,.0f}")
print(f"Ending bankroll:   ${combined_ending_bankroll:,.0f}")
print(f"Total P/L:         ${combined_profit_loss:,.0f}")
print(f"Combined return:   {combined_total_return:.2%}")
display(regime_backtest_summary_with_total)


Starting bankroll: $4,800,000
Ending bankroll:   $9,417,820
Total P/L:         $4,617,820
Combined return:   96.20%


,dataset_name,benchmark_ticker,test_start,test_end,allocation,ending_bankroll,profit_loss,total_return,annual_return,annual_volatility,...,sortino,max_drawdown,average_turnover,avg_active_names,avg_effective_names,avg_realized_max_weight,avg_beta_estimate,stress_worst_day,stress_worst_5_day,stress_beta_to_benchmark
0,regime_modern_tech_gain_2022_2026,XLK,2022-01-03,2026-05-21,1200000.0,2.625314e+06,1.425314e+06,1.187762,0.196352,0.276060,...,1.021568,-0.353088,0.046207,39.152838,25.312774,0.045592,0.941706,-0.066749,-0.145440,0.986897
1,regime_financial_crisis_loss_2005_2010,XLF,2005-01-03,2010-12-31,1200000.0,1.259663e+06,5.966259e+04,0.049719,0.008148,0.447407,...,0.024059,-0.775522,0.020582,33.000000,28.153795,0.039245,0.973596,-0.160466,-0.288043,1.014306
2,regime_nineties_volatility_1995_1999,MDY,1995-05-05,1999-12-31,1200000.0,3.507456e+06,2.307456e+06,1.922880,0.279594,0.161997,...,2.401461,-0.176761,0.042019,40.175439,23.546297,0.050110,0.613003,-0.065815,-0.100014,0.659096
3,regime_oil_pre2014_energy_2010_2013,XOP,2010-01-04,2013-12-31,1200000.0,2.025386e+06,8.253865e+05,0.687822,0.140629,0.253927,...,0.765372,-0.321159,0.016990,22.000000,21.029796,0.049358,0.765391,-0.092086,-0.204890,0.772409
4,COMBINED_4_REGIME_BANKROLL,mixed,1995-05-05,2026-05-21,4800000.0,9.417820e+06,4.617820e+06,0.962046,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
def mlflow_tracking_preflight(tracking_uri: str, timeout: float = 10.0) -> tuple[bool, str]:
    from urllib.error import HTTPError, URLError
    from urllib.parse import urlparse
    from urllib.request import Request, urlopen
    import os
    import socket

    parsed = urlparse(tracking_uri)
    if parsed.scheme not in {"http", "https"}:
        return True, "local tracking URI"
    if not parsed.hostname:
        return False, f"remote tracking URI has no hostname: {tracking_uri}"

    proxy_env = ["HTTPS_PROXY", "HTTP_PROXY", "ALL_PROXY", "https_proxy", "http_proxy", "all_proxy"]
    configured_proxy = next((name for name in proxy_env if os.environ.get(name)), None)
    if configured_proxy:
        try:
            request = Request(tracking_uri.rstrip("/") + "/", method="GET")
            with urlopen(request, timeout=timeout):
                return True, f"{parsed.hostname} reachable through {configured_proxy}"
        except HTTPError as exc:
            return True, f"{parsed.hostname} reachable through {configured_proxy}; HTTP {exc.code}"
        except URLError as exc:
            return False, f"{parsed.hostname} not reachable through {configured_proxy} ({exc.reason})"
        except OSError as exc:
            return False, f"{parsed.hostname} not reachable through {configured_proxy} ({exc})"

    port = parsed.port or (443 if parsed.scheme == "https" else 80)
    try:
        with socket.create_connection((parsed.hostname, port), timeout=timeout):
            return True, f"{parsed.hostname}:{port} reachable"
    except OSError as exc:
        return False, f"{parsed.hostname}:{port} not reachable ({exc})"


if LOG_TO_MLFLOW:
    mlflow_layout = init_mlflow(repo_root)
    tracking_ok, tracking_reason = mlflow_tracking_preflight(mlflow_layout["tracking_uri"])
    if not tracking_ok:
        message = f"Four-regime MLflow logging preflight failed: {tracking_reason}"
        if MLFLOW_REQUIRED:
            raise ConnectionError(message)
        print("Four-regime MLflow logging skipped:", message)
    else:
        import mlflow
        for dataset_name, payload in regime_backtest_runs.items():
            spec = payload["spec"]
            result_obj = payload["result"]
            portfolio_obj = payload["portfolio"]
            predictions_obj = payload["predictions"]
            risk_log_obj = payload["risk_log"]
            with start_run(
                run_name=f"{RUN_NAME}_four_regime_{dataset_name}",
                dataset_name=spec,
                tags={
                    "workflow": "hannah_four_regime_proxy_backtest",
                    "model_family": "torch_lstm_autoencoder",
                    "prediction_horizon": str(HORIZON),
                    "rebalance_frequency": portfolio_obj.metadata.get("rebalance_frequency", REBALANCE_FREQUENCY),
                    "benchmark_context_ticker": spec.benchmark_ticker,
                    "bankroll_allocation": str(REGIME_BACKTEST_ALLOCATION),
                },
                repo_root=repo_root,
            ):
                mlflow.log_params(
                    {
                        "model_name": MODEL_NAME,
                        "dataset_name": dataset_name,
                        "benchmark_ticker": spec.benchmark_ticker,
                        "test_start": str(spec.test_start),
                        "test_end": str(spec.test_end),
                        "horizon": HORIZON,
                        "score_col": portfolio_obj.metadata.get("score_col"),
                        "high_vol_max_weight": portfolio_obj.metadata.get("high_vol_max_weight"),
                        "normal_max_weight": portfolio_obj.metadata.get("normal_max_weight"),
                        "turnover_blend": portfolio_obj.metadata.get("turnover_blend"),
                        "beta_target": portfolio_obj.metadata.get("beta_target"),
                        "bankroll_allocation": REGIME_BACKTEST_ALLOCATION,
                    }
                )
                bankroll_row = regime_backtest_summary.loc[regime_backtest_summary["dataset_name"] == dataset_name].iloc[0]
                mlflow.log_metrics(
                    {
                        "bankroll_allocation": float(bankroll_row["allocation"]),
                        "ending_bankroll": float(bankroll_row["ending_bankroll"]),
                        "profit_loss": float(bankroll_row["profit_loss"]),
                        "avg_active_names": float(bankroll_row["avg_active_names"]),
                        "avg_effective_names": float(bankroll_row["avg_effective_names"]),
                        "avg_realized_max_weight": float(bankroll_row["avg_realized_max_weight"]),
                        "avg_beta_estimate": float(bankroll_row["avg_beta_estimate"]),
                    }
                )
                log_predictions(predictions_obj)
                log_portfolio(portfolio_obj)
                log_backtest(result_obj)
                mlflow.log_artifact(str(payload["output_dir"] / "risk_log.parquet"))
                mlflow.log_artifact(str(payload["output_dir"] / "stress_table.parquet"))
        with start_run(
            run_name=f"{RUN_NAME}_four_regime_combined_bankroll",
            dataset_name=REGIME_BACKTEST_DATASETS[0],
            tags={
                "workflow": "hannah_four_regime_proxy_backtest_summary",
                "combined_backtest": "true",
                "total_bankroll": str(REGIME_BACKTEST_TOTAL_BANKROLL),
            },
            repo_root=repo_root,
        ):
            mlflow.log_metrics(
                {
                    "total_bankroll": REGIME_BACKTEST_TOTAL_BANKROLL,
                    "combined_ending_bankroll": combined_ending_bankroll,
                    "combined_profit_loss": combined_profit_loss,
                    "combined_total_return": combined_total_return,
                }
            )
            mlflow.log_artifact(str(summary_path))
            mlflow.log_artifact(str(csv_summary_path))
        print("Four-regime MLflow logging complete.")
else:
    print("Four-regime MLflow logging skipped because SKIP_MLFLOW=1.")


🏃 View run Hannah_Multi_Regime_Volatility_Router_four_regime_regime_modern_tech_gain_2022_2026 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/10/runs/45c72fba78614d7dab783921d0d6648c
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/10
🏃 View run Hannah_Multi_Regime_Volatility_Router_four_regime_regime_financial_crisis_loss_2005_2010 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/11/runs/ea447d89f9184d479bb9417366ba382c
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/11
🏃 View run Hannah_Multi_Regime_Volatility_Router_four_regime_regime_nineties_volatility_1995_1999 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/12/runs/03310dccb1a04c5eb00a93ee744102b3
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/12
🏃 View run Hannah_Multi_Regime_Volatility_Router_four_regime_regime_oil_pre2014_energy_2010_2013 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/ex